<p class="lead">Este notebook desarrolla la quinta unidad del curso, en R. Las diapositivas de clase son un mapa; este texto es el territorio. Las Unidades 2–4 entrenaron modelos con datos cuya respuesta se conocía: cada aspirado traía un resultado de biopsia, cada imagen un diagnóstico, y la calidad de un modelo era la concordancia entre sus predicciones y esas etiquetas. Esta unidad elimina las etiquetas. Lo que queda es una matriz de datos y tres preguntas que se le pueden hacer sin ninguna clave de respuestas: <em>¿los casos se reparten en grupos?</em> (agrupamiento o clustering — k-means y agrupamiento jerárquico), <em>¿pueden las muchas características medidas resumirse en unas pocas?</em> (reducción de dimensionalidad — análisis de componentes principales) y <em>¿pueden separarse señales mezcladas en sus fuentes?</em> (separación ciega de fuentes — análisis de componentes independientes). Cada método se deriva, se sigue a mano en un ejemplo del tamaño de una hoja de papel, se confirma en R, se ejercita con los datos de cáncer de mama que reaparecen a lo largo del curso y se lleva al límite hasta que falla, porque saber <em>cuándo</em> falla un método es la diferencia entre usarlo y ser usado por él. El Laboratorio 3 aplica estos cuatro métodos; el Taller 3 es un cuestionario sobre ellos; este texto es la preparación para ambos.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Definir el aprendizaje no supervisado y distinguir entre agrupamiento, reducción de dimensionalidad y estimación de densidad; explicar qué puede y qué no puede decirte un resultado de agrupamiento en ausencia de etiquetas.</li>
<li>Enunciar la función objetivo de k-means (suma de cuadrados intra-cluster), ejecutar a mano el algoritmo de Lloyd sobre un conjunto de datos pequeño y explicar por qué converge a un mínimo local, no global.</li>
<li>Describir las principales variantes de k-means (fuzzy c-means, k-medoids, mini-batch, k-means++, esférico) e indicar cuándo es apropiada cada una.</li>
<li>Elegir el número de clusters con el método del codo, la silueta y el estadístico gap, y nombrar las situaciones en que k-means falla (tamaños desiguales, formas no convexas, características sin escalar).</li>
<li>Ejecutar a mano el agrupamiento jerárquico aglomerativo, enunciar las reglas de enlace simple, completo, promedio y de Ward, leer un dendrograma, cortarlo y calcular la correlación cofenética.</li>
<li>Derivar el análisis de componentes principales como maximización de la varianza y como mínimo error de reconstrucción; calcular a mano las componentes principales de un conjunto de datos pequeño a partir de la matriz de covarianza; interpretar un gráfico de sedimentación (scree plot), las cargas (loadings) y un biplot; explicar por qué la estandarización va primero.</li>
<li>Explicar el análisis de componentes independientes como separación ciega de fuentes guiada por la no gaussianidad, implementar FastICA y usarlo para eliminar un artefacto de un registro fisiológico multicanal.</li>
<li>Evaluar un resultado no supervisado con índices internos y externos, y reconocer la validación circular.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Cada bloque de código (chunk) es breve y está comentado. Ejecútalo, lee su salida y luego lee el párrafo que la interpreta. Las cajas azules contienen ideas clave, las naranjas errores frecuentes y las moradas conectan esta unidad con otras unidades o con el curso de bioinformática (BGM324). El notebook usa solo R base y los paquetes <code>mlbench</code>, <code>cluster</code>, <code>MASS</code> y <code>ggplot2</code>, todos preinstalados en este entorno; en un entorno de ejecución de R recién creado en Google Colab, el estudiante ejecuta <code>install.packages(c("mlbench","cluster"))</code> una vez. No se descarga ningún dato. Tres algoritmos se siguen a mano (k-means en la Sección 2.4, agrupamiento aglomerativo en la Sección 3.3, PCA en la Sección 4.2) antes de que el código los confirme, y dos algoritmos que no tienen función en R base —<strong>FastICA</strong> (Sección 5) y el <strong>agrupamiento espectral</strong> (Sección 6)— se implementan desde cero en una docena de líneas cada uno, tal como la edición Python escribió los suyos. Se da por conocido, y se usa sin volver a derivarlo, el vocabulario del Suplemento S1: división entrenamiento/prueba, validación cruzada, exactitud, fuga de datos.</p></div>

<div class="box warn"><h4>Una nota sobre el conjunto de datos, respecto de la edición Python</h4><p>La edición Python de esta unidad usó los datos Wisconsin Diagnostic Breast Cancer de scikit-learn, con 30 características. El paquete <code>mlbench</code> de R incluye el conjunto Wisconsin de cáncer de mama más antiguo, de <strong>9 características</strong> (las nueve puntuaciones citológicas, cada una en una escala ordinal de 1–10). Como las nueve características ya comparten una misma escala, varios números difieren de los de la edición Python y, en particular, la estandarización cambia el resultado mucho menos que allí — un contraste que esta unidad aprovecha en lugar de ocultarlo (Secciones 2.8 y 4.3). Donde la edición Python se apoyaba en conjuntos de datos sin equivalente en R —los dígitos manuscritos y la tabla de química de vinos—, esta edición <em>simula</em> un equivalente y lo indica en el punto en que se usa.</p></div>



In [ ]:
# Bibliotecas usadas en todo el notebook. En un entorno de R recién creado en Colab: install.packages(c("mlbench","cluster")).
suppressPackageStartupMessages({ library(mlbench); library(cluster); library(MASS) })
set.seed(0)                                  # todos los números citados más abajo son reproducibles
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#7f7f7f"; PURPLE <- "#7b4f9d"
PALETTE <- c(BLUE, ORANGE, GREEN, PURPLE, "#b5a000", "#9b2b2b", GREY)

# El índice de Rand ajustado (Sección 2.8), usado en muchos bloques de código posteriores, se define aquí una sola vez.
adjusted_rand <- function(a, b) {
  tab <- table(a, b); n <- sum(tab); sc <- function(x) sum(choose(x, 2))
  ai <- rowSums(tab); bj <- colSums(tab)
  idx <- sc(as.vector(tab)); exp_idx <- sc(ai) * sc(bj) / choose(n, 2); max_idx <- (sc(ai) + sc(bj)) / 2
  (idx - exp_idx) / (max_idx - exp_idx)
}
cat("R", as.character(getRversion()), "| mlbench, cluster, MASS loaded | ARI helper ready\n")



La salida impresa confirma el entorno. `set.seed(0)` fija todos los sorteos aleatorios, de modo que las figuras y los números del texto son exactamente los que produce el código; la función auxiliar `adjusted_rand` se define una vez y se reutiliza.

## 1. Aprender sin etiquetas

### 1.1 Qué cambia cuando se eliminan las etiquetas

En todas las unidades anteriores, la muestra de entrenamiento era un par \( (\mathbf{X}, \mathbf{y}) \): una matriz de datos de \(n\) casos descritos por \(p\) características y un vector de \(n\) respuestas conocidas. El algoritmo de aprendizaje ajustaba un modelo hasta que sus salidas concordaban con \(\mathbf{y}\), y el Suplemento S1 explicó cómo medir esa concordancia en casos que el modelo no había visto. Esto es aprendizaje <strong>supervisado</strong>, llamado así por el profesor que proporciona las respuestas.

El <strong>aprendizaje no supervisado</strong> parte solo de \(\mathbf{X}\). No hay \(\mathbf{y}\); por lo tanto, no hay error que minimizar frente a una verdad y, por lo tanto, no hay exactitud, sensibilidad ni curva ROC. Lo que el algoritmo tiene en su lugar es la <em>estructura</em> de \(\mathbf{X}\): qué casos se parecen a cuáles, qué características varían juntas, qué direcciones del espacio de características concentran la mayor parte de la variación. Las tres tareas principales son:

| Tarea | Pregunta que se le hace a \(\mathbf{X}\) | Salida | Métodos en esta unidad |
|---|---|---|---|
| **Agrupamiento (clustering)** | ¿Se reparten los \(n\) casos en grupos de casos mutuamente similares? | Una partición de las filas (una etiqueta por caso, inventada por el algoritmo) | k-means (Sección 2), agrupamiento jerárquico (Sección 3), mezclas gaussianas, DBSCAN, agrupamiento espectral (Sección 6) |
| **Reducción de dimensionalidad** | ¿Pueden reemplazarse las \(p\) características por \(q \ll p\) nuevas que conserven la mayor parte de la información? | Una transformación de las columnas (nuevas coordenadas por caso) | PCA (Sección 4), ICA (Sección 5); t-SNE y UMAP para visualización (Sección 4.7) |
| **Estimación de densidad** | ¿Cuál es la distribución de probabilidad que generó las filas? | Un modelo \(p(\mathbf{x})\) que asigna una densidad de probabilidad a cualquier punto | Modelos de mezclas gaussianas (Sección 6); los histogramas y las estimaciones de densidad por kernel de la Unidad 3 |

Las tres están relacionadas —un modelo de mezcla es una estimación de densidad cuyas componentes son clusters; el agrupamiento es la reducción de dimensionalidad más agresiva, de \(p\) características a una sola variable categórica— y todas buscan una <em>descripción compacta</em> de los datos: unos pocos centroides, unos pocos ejes, unas pocas componentes. La compacidad es lo que hace útil la descripción y lo que la hace perder información.

### 1.2 Qué puede y qué no puede decirte el aprendizaje no supervisado

Un método no supervisado siempre devuelve una respuesta. Si se le piden tres clusters, k-means devuelve tres clusters tanto si los datos tienen tres grupos como si tienen dos o ninguno; si se le piden dos componentes, PCA devuelve dos tanto si explican el 95 % como el 15 % de la varianza. Esta es la dificultad metodológica central de la unidad: <strong>la salida no puede validarse frente a una verdad, porque no se proporcionó ninguna verdad</strong>. Lo que sí puede hacerse, y lo que desarrolla la Sección 7, es medir propiedades internas (¿son los clusters compactos y separados? ¿cuánta varianza retienen las componentes?), medir la estabilidad (¿cambia el resultado cuando se perturba la muestra?) y —cuando existen etiquetas pero se ocultaron, como en todos los ejemplos de este notebook— comparar el resultado con ellas <em>a posteriori</em>, como una verificación externa que nunca estuvo disponible para el algoritmo.

El último punto merece cuidado. Los datos de cáncer de mama<sup class="c"><a href="#r1">[1]</a></sup> traen una etiqueta de biopsia, y este notebook la ocultará, agrupará las características nucleares y luego preguntará si los clusters coinciden con el diagnóstico. Cuando coinciden, la tentación es decir que k-means "diagnosticó" los tumores. No lo hizo: encontró que los núcleos malignos y benignos difieren lo suficiente en tamaño y forma como para que los dos grupos más compactos de los datos resulten separarlos. Si la fuente dominante de variación hubiera sido, por ejemplo, el laboratorio que preparó la lámina, k-means habría encontrado laboratorios, con la misma seguridad. La Unidad 2 hizo esta corrección una vez y se repite aquí porque el error es común en la literatura aplicada: <em>el agrupamiento descubre estructura; no descubre la estructura que esperabas encontrar</em>.<sup class="c"><a href="#r2">[2]</a></sup>

¿Por qué, entonces, aprender sin etiquetas? Porque las etiquetas son caras (una biopsia, una hora de un patólogo, un seguimiento de diez años), porque puede que aún no existan (una enfermedad cuyos subtipos se desconocen — Sección 8), porque la pregunta puede ser exploratoria y porque el preprocesamiento de un problema supervisado —reducir características correlacionadas a unas pocas no correlacionadas— es en sí mismo no supervisado. Solo el agrupamiento tiene una literatura de miles de algoritmos;<sup class="c"><a href="#r3">[3]</a></sup> esta unidad presenta las dos familias que concentran la mayor parte de su uso y las dos descomposiciones que concentran la mayor parte de la reducción de dimensionalidad.

### 1.3 El ejemplo conductor, con las etiquetas ocultas

El bloque de código siguiente carga los datos Wisconsin de cáncer de mama desde <code>mlbench</code>,<sup class="c"><a href="#r4">[4]</a></sup> descarta el identificador y los registros incompletos, convierte las nueve puntuaciones ordinales a numéricas, las estandariza (la Sección 2.8 muestra qué ocurre sin este paso) y aparta el diagnóstico en una variable que ningún algoritmo verá hasta la Sección 2.8. La figura grafica dos características tal como las ve el algoritmo —sin etiquetas— y, a su lado, tal como las vería un método supervisado.



In [ ]:
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # descarta la columna Id
bc <- bc[complete.cases(bc), ]                             # descarta las 16 filas con valores faltantes
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # ordinal -> numérica
X_raw <- as.matrix(bc[, 1:9]); y_hidden <- bc$Class        # etiqueta usada SOLO para verificar resultados, nunca para ajustar
X_std <- scale(X_raw)                                      # cada característica a media 0 y desviación estándar 1
cat("Data matrix:", nrow(X_raw), "x", ncol(X_raw),
    "| after standardisation max|mean| =", round(max(abs(colMeans(X_std))), 12),
    ", min sd =", round(min(apply(X_std, 2, sd)), 6), "\n")
print(table(diagnosis = y_hidden))

par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(jitter(X_raw[, "Cell.size"]), jitter(X_raw[, "Cl.thickness"]), col = GREY, pch = 19, cex = 0.5,
     xlab = "cell size", ylab = "clump thickness", main = "what an unsupervised method sees")
cols <- ifelse(y_hidden == "malignant", ORANGE, BLUE)
plot(jitter(X_raw[, "Cell.size"]), jitter(X_raw[, "Cl.thickness"]), col = cols, pch = 19, cex = 0.5,
     xlab = "cell size", ylab = "clump thickness", main = "the same cases with the hidden label")
legend("bottomright", c("benign", "malignant"), col = c(BLUE, ORANGE), pch = 19, bty = "n")



<p class="caption"><b>Figura 1.</b> Izquierda: 683 aspirados descritos por el tamaño celular y el grosor del cúmulo (clump thickness), tal como los recibe un algoritmo no supervisado: una sola nube gris (con jitter, porque las puntuaciones son enteras), más densa en la parte inferior izquierda. Derecha: los mismos puntos coloreados según el resultado de biopsia que se ha ocultado. Los núcleos malignos (naranja) se concentran en valores altos de tamaño celular y de grosor del cúmulo, y los benignos (azul) en valores bajos; la separación es más clara que en la vista de dos características de la edición Python porque estas nueve puntuaciones citológicas fueron elegidas por un patólogo precisamente para separar las clases. Si un algoritmo de agrupamiento dividirá la nube según el diagnóstico, según alguna otra dirección o si no la dividirá en absoluto, es la pregunta de esta unidad.</p>

La salida impresa registra la curación de los datos: 683 casos completos (se descartaron las 16 filas con un valor faltante de <em>Bare.nuclei</em>), 444 benignos y 239 malignos, y cada columna estandarizada con media 0 y desviación estándar 1. La estandarización importa porque todos los métodos de esta unidad se basan en distancias euclidianas o en varianzas, y ambas quedan dominadas por la característica que tenga los números más grandes. Aquí las nueve características ya comparten la escala 1–10, de modo que el efecto es leve; la Sección 2.8 y la Sección 4.3 miden exactamente cuán leve, y eso mismo es la lección.

## 2. k-means

### 2.1 Definición: la suma de cuadrados intra-cluster

Dados \(n\) puntos \(\mathbf{x}_1, \dots, \mathbf{x}_n \in \mathbb{R}^p\) y un número elegido de clusters \(k\), k-means busca una partición de los puntos en \(k\) conjuntos disjuntos \(C_1, \dots, C_k\) y \(k\) centroides \(\boldsymbol{\mu}_1, \dots, \boldsymbol{\mu}_k\) que minimicen la <strong>suma de cuadrados intra-cluster</strong> (WCSS, within-cluster sum of squares; la función <code>kmeans</code> de R la informa como <code>tot.withinss</code>):

<div class="eq">\[ J(C, \boldsymbol{\mu}) = \sum_{j=1}^{k} \sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert_2^2 \]
<span class="where">donde \(\lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert_2^2\) es la distancia euclidiana al cuadrado entre un punto y el centroide del cluster al que pertenece. Para una partición fija, \(J\) se minimiza cuando cada \(\boldsymbol{\mu}_j\) es la media aritmética de los puntos de \(C_j\) —de ahí el nombre— y, para centroides fijos, \(J\) se minimiza cuando cada punto se asigna a su centroide más cercano.</span></div>

La función objetivo admite una segunda lectura que explica en qué es bueno k-means. Desarrollando los cuadrados, \(J\) es proporcional a la suma, sobre los clusters, de la <em>distancia cuadrática total entre pares dentro del cluster</em> dividida por el tamaño del cluster; por lo tanto, k-means busca grupos de puntos mutuamente cercanos, medidos en distancia euclidiana al cuadrado. De ello se sigue que el método prefiere clusters redondos (isotrópicos), de dispersión similar y de tamaño similar: las tres preferencias que la Sección 2.7 convierte en modos de falla.

Encontrar exactamente la partición que minimiza \(J\) es NP-difícil, incluso para \(k = 2\) en dimensión general, de modo que todo algoritmo práctico es una heurística que encuentra un mínimo <em>local</em>. El nombre "k-means" lo acuñó MacQueen en 1967 para una versión en línea (online) del procedimiento;<sup class="c"><a href="#r5">[5]</a></sup> el algoritmo por lotes que todos usan fue descrito por Lloyd en un informe de 1957 de los Bell Laboratories sobre modulación por impulsos codificados, publicado recién en 1982,<sup class="c"><a href="#r6">[6]</a></sup> y se conoce como <strong>algoritmo de Lloyd</strong>. La Sección 2.3 lo enuncia; la Sección 2.4 lo ejecuta a mano.

### 2.2 Tipos de k-means

El algoritmo de libro de texto tiene varias variantes, cada una de las cuales aborda una de sus limitaciones. La clase las enumera bajo "tipos"; aquí va un párrafo para cada una, con la situación que la justifica.

**k-means duro (el estándar).** Cada punto pertenece exactamente a un cluster. Esto es lo que implementa <code>stats::kmeans</code> de R y lo que usa el resto de esta sección. Úsalo cuando se espera que los clusters sean compactos, aproximadamente esféricos y de tamaño comparable, y cuando lo que se necesita es una partición nítida.

**k-means blando o difuso (fuzzy c-means).** Cada punto recibe un <em>grado de pertenencia</em> \(u_{ij} \in [0, 1]\) en cada cluster, con \(\sum_j u_{ij} = 1\), y la función objetivo pasa a ser \(\sum_i \sum_j u_{ij}^m \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert^2\) con un exponente de difusividad \(m > 1\) (normalmente 2). Los centroides son medias ponderadas por la pertenencia. La monografía de Bezdek de 1981 es la referencia estándar.<sup class="c"><a href="#r7">[7]</a></sup> Úsalo cuando los puntos cercanos a una frontera no deben forzarse a un solo grupo: la segmentación de tejidos en imágenes de resonancia magnética, donde un vóxel puede contener tanto sustancia gris como blanca, es la aplicación biomédica clásica. La función <code>e1071::cmeans</code> lo implementa; el modelo de mezcla gaussiana de la Sección 6, que entrega probabilidades posteriores de pertenencia, es la versión probabilística fundamentada de la misma idea.

**k-medoids (PAM).** El representante de cada cluster no es la media sino un <em>medoide</em> —el punto de datos real cuya distancia total a los demás miembros es la menor— y la función objetivo puede usar cualquier disimilitud, no solo la distancia euclidiana al cuadrado. Partitioning Around Medoids (PAM) es el algoritmo de Kaufman y Rousseeuw,<sup class="c"><a href="#r8">[8]</a></sup> disponible en R como <code>cluster::pam</code>. Úsalo cuando los datos no son euclidianos (distancias de alineamiento de secuencias; disimilitudes entre pacientes construidas a partir de variables mixtas categóricas y numéricas), cuando importa la robustez frente a valores atípicos (un medoide no puede ser arrastrado por un único punto extremo) o cuando el representante debe ser un caso real que un clínico pueda examinar. Su costo es \(O(k(n-k)^2)\) por iteración, por lo que necesita aproximaciones (<code>cluster::clara</code>) para \(n\) grande.

**k-means mini-batch.** Cada iteración actualiza los centroides con un pequeño subconjunto aleatorio de los datos en lugar de la muestra completa, usando una tasa de aprendizaje por centroide que disminuye con el número de puntos ya asignados a él.<sup class="c"><a href="#r9">[9]</a></sup> Úsalo cuando \(n\) está en los millones (píxeles de una imagen de lámina completa, lecturas de una corrida de secuenciación) y un objetivo ligeramente peor es un precio aceptable por una aceleración de uno o dos órdenes de magnitud.

**Inicialización k-means++.** No es un objetivo distinto sino un <em>inicio</em> distinto: el primer centroide es un punto de datos aleatorio y cada centroide siguiente se sortea con probabilidad proporcional a su distancia al cuadrado desde el centroide más cercano ya elegido, de modo que los centroides iniciales quedan dispersos. Arthur y Vassilvitskii demostraron que esta siembra por sí sola da un objetivo esperado dentro de un factor \(O(\log k)\) del óptimo y, en la práctica, elimina la mayoría de los malos mínimos locales en que cae la inicialización aleatoria simple.<sup class="c"><a href="#r10">[10]</a></sup> La función <code>kmeans</code> de R base no la incluye, así que la Sección 2.5 la implementa en unas pocas líneas y mide lo que aporta.

**k-means esférico.** Cada punto se normaliza primero a longitud unitaria y la disimilitud es la distancia coseno \(1 - \cos\theta\) en lugar de la distancia euclidiana, de modo que solo importa la <em>dirección</em> de un vector de características, no su magnitud. Úsalo cuando el total es arbitrario pero el perfil es informativo: perfiles de expresión génica con profundidad de secuenciación variable, espectros de distinta intensidad global, conteos de palabras. En R, normaliza cada fila a longitud unitaria antes de aplicar <code>kmeans</code>.

### 2.3 El algoritmo de Lloyd

<ol>
<li><strong>Inicializar.</strong> Elegir \(k\) centroides iniciales \(\boldsymbol{\mu}_1^{(0)}, \dots, \boldsymbol{\mu}_k^{(0)}\): clásicamente, \(k\) puntos de datos elegidos al azar; hoy, mediante k-means++.</li>
<li><strong>Paso de asignación.</strong> Asignar cada punto al cluster de su centroide más cercano: \(C_j \leftarrow \{\mathbf{x}_i : \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert \le \lVert \mathbf{x}_i - \boldsymbol{\mu}_l \rVert \;\forall\, l\}\).</li>
<li><strong>Paso de actualización.</strong> Recalcular cada centroide como la media de su cluster: \(\boldsymbol{\mu}_j \leftarrow \frac{1}{|C_j|}\sum_{\mathbf{x}_i \in C_j} \mathbf{x}_i\).</li>
<li><strong>Repetir</strong> los pasos 2–3 hasta que ninguna asignación cambie (o los centroides se muevan menos que una tolerancia, o se alcance un número máximo de iteraciones).</li>
</ol>

Cada paso solo puede disminuir \(J\) o dejarlo igual —el paso de asignación porque cada punto se mueve a un centroide más cercano, el paso de actualización porque la media es el punto que minimiza la suma de distancias al cuadrado a un conjunto— y, como hay un número finito de particiones, el algoritmo termina. Donde termina es un mínimo local que depende del inicio, que es el tema de la Sección 2.5. El costo de una iteración es \(O(nkp)\): cada punto se compara con cada centroide en cada dimensión.

### 2.4 Ejemplo resuelto a mano

Seis puntos bidimensionales y \(k = 2\). Las coordenadas son enteras para que cada distancia pueda calcularse en papel; se usa la distancia al cuadrado en todo momento porque el centroide más cercano es el mismo tanto si se compara la distancia como su cuadrado, y los cuadrados evitan las raíces.

| punto | \(x_1\) | \(x_2\) |
|---|---|---|
| \(\mathbf{x}_1\) | 1 | 1 |
| \(\mathbf{x}_2\) | 2 | 1 |
| \(\mathbf{x}_3\) | 1 | 2 |
| \(\mathbf{x}_4\) | 6 | 5 |
| \(\mathbf{x}_5\) | 7 | 5 |
| \(\mathbf{x}_6\) | 6 | 6 |

**Inicialización.** Deliberadamente mala, para mostrar cómo se recupera el algoritmo: \(\boldsymbol{\mu}_1^{(0)} = \mathbf{x}_1 = (1, 1)\) y \(\boldsymbol{\mu}_2^{(0)} = \mathbf{x}_2 = (2, 1)\), dos puntos vecinos del mismo grupo visual.

**Iteración 1 — asignación.** Distancias al cuadrado \(d^2(\mathbf{x}_i, \boldsymbol{\mu}_j) = (x_{i1} - \mu_{j1})^2 + (x_{i2} - \mu_{j2})^2\):

| punto | \(d^2\) a \(\boldsymbol{\mu}_1 = (1, 1)\) | \(d^2\) a \(\boldsymbol{\mu}_2 = (2, 1)\) | asignado a |
|---|---|---|---|
| \((1, 1)\) | \(0\) | \(1\) | \(C_1\) |
| \((2, 1)\) | \(1\) | \(0\) | \(C_2\) |
| \((1, 2)\) | \(1\) | \(2\) | \(C_1\) |
| \((6, 5)\) | \(41\) | \(32\) | \(C_2\) |
| \((7, 5)\) | \(52\) | \(41\) | \(C_2\) |
| \((6, 6)\) | \(50\) | \(41\) | \(C_2\) |

El objetivo en este punto es la suma de las distancias al cuadrado elegidas: \(J = 0 + 0 + 1 + 32 + 41 + 41 = 115\).

**Iteración 1 — actualización.** \(\boldsymbol{\mu}_1 = \) media de \(\{(1,1), (1,2)\} = (1, 1.5)\); \(\boldsymbol{\mu}_2 = \) media de \(\{(2,1), (6,5), (7,5), (6,6)\} = (5.25, 4.25)\).

**Iteración 2 — asignación.** El punto \(\mathbf{x}_2 = (2,1)\) está ahora a \(1.25\) de \(\boldsymbol{\mu}_1\) pero a \(21.125\) de \(\boldsymbol{\mu}_2\), así que se mueve a \(C_1\); los tres puntos lejanos permanecen en \(C_2\). El objetivo baja a \(J = 0.25 + 1.25 + 0.25 + 1.125 + 3.625 + 3.625 = 10.125\).

**Iteración 2 — actualización.** \(\boldsymbol{\mu}_1 = \) media de \(\{(1,1), (2,1), (1,2)\} = (\tfrac{4}{3}, \tfrac{4}{3}) \approx (1.33, 1.33)\); \(\boldsymbol{\mu}_2 = \) media de \(\{(6,5), (7,5), (6,6)\} = (\tfrac{19}{3}, \tfrac{16}{3}) \approx (6.33, 5.33)\).

Un tercer paso de asignación no cambiaría nada, de modo que el algoritmo ha <strong>convergido</strong> tras dos iteraciones. El objetivo final es \(J^\ast = \tfrac{24}{9} = \tfrac{8}{3} \approx 2.667\). La secuencia \(115 \to 10.125 \to 2.667\) es la disminución monótona prometida en la Sección 2.3. El bloque de código siguiente implementa el algoritmo de Lloyd en una docena de líneas de R base e imprime las mismas tablas.



In [ ]:
# Algoritmo de Lloyd en R base, seguido sobre los seis puntos resueltos a mano con el mismo inicio (malo).
P <- matrix(c(1,1, 2,1, 1,2, 6,5, 7,5, 6,6), ncol = 2, byrow = TRUE)
lloyd <- function(P, centroids, max_iter = 20, verbose = TRUE) {
  history <- numeric(0)
  for (it in seq_len(max_iter)) {
    d2  <- sapply(seq_len(nrow(centroids)), function(j) rowSums(sweep(P, 2, centroids[j, ])^2))
    lab <- max.col(-d2, ties.method = "first")                      # paso de asignación: centroide más cercano
    history <- c(history, sum(d2[cbind(seq_len(nrow(P)), lab)]))    # objetivo J tras la asignación
    if (verbose) {
      tab <- data.frame(round(d2, 3)); names(tab) <- paste0("d2_mu", seq_len(ncol(d2)))
      tab$assigned <- lab; rownames(tab) <- paste0("x", seq_len(nrow(P)))
      cat(sprintf("iteration %d: J = %.3f\n", it, history[it])); print(tab)
    }
    new <- t(sapply(seq_len(nrow(centroids)), function(j) colMeans(P[lab == j, , drop = FALSE])))
    if (all(abs(new - centroids) < 1e-12)) break                    # paso de actualización; se detiene cuando es estable
    centroids <- new
  }
  list(labels = lab, centroids = centroids, history = history)
}
res <- lloyd(P, P[c(1, 2), ])
cat("\nfinal centroids:", paste(round(res$centroids, 3), collapse = " "),
    "| J sequence:", paste(round(res$history, 3), collapse = " -> "), "\n")



La salida impresa reproduce ambas tablas hechas a mano —incluido el cambio de \(\mathbf{x}_2\) en la iteración 2— y la secuencia del objetivo \(115 \to 10.125 \to 2.667\); el bucle termina en la tercera pasada, la verificación de convergencia en la que ningún centroide se mueve. El bloque de código siguiente entrega a la propia función <code>kmeans</code> de R los mismos dos centroides iniciales y dibuja la trayectoria.



In [ ]:
km_hand <- kmeans(P, centers = P[c(1, 2), ], algorithm = "Lloyd", iter.max = 20)
cat("kmeans() centroids:", paste(round(km_hand$centers, 3), collapse = " "),
    "| tot.withinss (J*):", round(km_hand$tot.withinss, 3), "\n")
# posiciones de los centroides tras 0, 1 y 2 actualizaciones, para la figura
path <- list(P[c(1, 2), ])
for (t in 1:2) {
  d2 <- sapply(1:2, function(j) rowSums(sweep(P, 2, path[[t]][j, ])^2)); lab <- max.col(-d2)
  path[[t + 1]] <- t(sapply(1:2, function(j) colMeans(P[lab == j, , drop = FALSE])))
}
par(mar = c(4, 4, 2, 1))
plot(P, pch = 19, cex = 1.6, col = PALETTE[res$labels], xlim = c(0, 8), ylim = c(0, 7),
     xlab = expression(x[1]), ylab = expression(x[2]), main = "Lloyd's algorithm: centroids after 0, 1, 2 updates")
text(P[, 1], P[, 2], paste0("x", 1:6), pos = 1, cex = 0.8)
for (j in 1:2) {
  cx <- sapply(path, function(m) m[j, 1]); cy <- sapply(path, function(m) m[j, 2])
  lines(cx, cy, lty = 2, col = c(BLUE, ORANGE)[j])
  points(cx, cy, pch = 4, lwd = 2, cex = c(1, 1.4, 2), col = c(BLUE, ORANGE)[j])
}
legend("topleft", c("cluster C1", "cluster C2"), col = c(BLUE, ORANGE), pch = 19, bty = "n")



<p class="caption"><b>Figura 2.</b> Los seis puntos coloreados según su cluster final, y los dos centroides (cruces, más grandes a medida que avanzan las iteraciones) desplazándose desde el inicio malo \(\boldsymbol{\mu}_1^{(0)} = (1,1)\), \(\boldsymbol{\mu}_2^{(0)} = (2,1)\) hasta las posiciones finales \((1.33, 1.33)\) y \((6.33, 5.33)\). El centroide naranja recorre una gran distancia: tras la primera asignación, los tres puntos lejanos lo arrastran al centro del plano, y solo la segunda iteración lo lleva al grupo de la derecha. La función <code>kmeans</code> de R informa los mismos centroides y el mismo objetivo \(8/3 \approx 2.667\) que la implementación desde cero.</p>

### 2.5 Mínimos locales e inicialización

El algoritmo de Lloyd desciende hasta el mínimo local de \(J\) más cercano, y \(J\) tiene muchos. El mal resultado clásico es que dos centroides compartan un mismo grupo real mientras otro grupo real queda dividido o es absorbido por un vecino; el algoritmo no puede escapar, porque mover un centroide a través del hueco entre grupos primero <em>aumentaría</em> \(J\). Los remedios estándar son (i) un mejor inicio, k-means++, y (ii) los <em>reinicios</em>: ejecutar el algoritmo desde varias inicializaciones y quedarse con la solución de menor \(J\) (el argumento <code>nstart</code> de R). El bloque de código siguiente genera cinco grupos gaussianos bien separados y luego compara doce ejecuciones individuales desde inicios aleatorios simples con doce desde una siembra k-means++ implementada desde cero.



In [ ]:
make_blobs <- function(n, centers, sd = 1.2) {           # sustituto en R base de make_blobs de sklearn
  k <- nrow(centers); per <- rep(n %/% k, k); if (n %% k) per[seq_len(n %% k)] <- per[seq_len(n %% k)] + 1
  list(X = do.call(rbind, lapply(seq_len(k), function(j)
         cbind(rnorm(per[j], centers[j, 1], sd), rnorm(per[j], centers[j, 2], sd)))),
       y = rep(seq_len(k), per))
}
kpp_init <- function(X, k, seed) {                        # siembra k-means++, desde cero
  set.seed(seed); cent <- X[sample(nrow(X), 1), , drop = FALSE]
  for (j in 2:k) { d2 <- apply(X, 1, function(x) min(colSums((t(cent) - x)^2)))
                   cent <- rbind(cent, X[sample(nrow(X), 1, prob = d2), , drop = FALSE]) }
  cent
}
set.seed(0)
centres5 <- matrix(c(0,0, 9,0, 0,9, 9,9, 4.5,16), ncol = 2, byrow = TRUE)
b <- make_blobs(500, centres5, 1.2); Xb <- b$X
rand_J <- sapply(1:12, function(s) { set.seed(s); kmeans(Xb, 5, nstart = 1, algorithm = "Lloyd", iter.max = 50)$tot.withinss })
kpp_J  <- sapply(1:12, function(s) { set.seed(s); kmeans(Xb, kpp_init(Xb, 5, s), nstart = 1, iter.max = 50)$tot.withinss })
best <- min(c(rand_J, kpp_J)); thr <- 1.2 * best        # "atascada" = objetivo más de un 20% sobre el mejor encontrado
print(rbind(`random init`    = round(rand_J, 0), `k-means++ init` = round(kpp_J, 0)))
cat(sprintf("\nbest J = %.0f; runs stuck in a clearly worse local minimum: random %d/12, k-means++ %d/12\n",
            best, sum(rand_J > thr), sum(kpp_J > thr)))

worst <- which.max(rand_J)
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
set.seed(worst); bad <- kmeans(Xb, 5, nstart = 1, algorithm = "Lloyd", iter.max = 50)
good <- kmeans(Xb, kpp_init(Xb, 5, 1), nstart = 1, iter.max = 50)
plot(Xb, col = PALETTE[bad$cluster], pch = 19, cex = 0.5, asp = 1, xlab = "", ylab = "",
     main = sprintf("random start, J = %.0f", bad$tot.withinss)); points(bad$centers, pch = 4, lwd = 2, cex = 1.6)
plot(Xb, col = PALETTE[good$cluster], pch = 19, cex = 0.5, asp = 1, xlab = "", ylab = "",
     main = sprintf("k-means++, J = %.0f", good$tot.withinss)); points(good$centers, pch = 4, lwd = 2, cex = 1.6)



<p class="caption"><b>Figura 3.</b> Izquierda: una mala ejecución con inicio aleatorio (semilla 2). Dos centroides comparten un mismo grupo y un centroide queda entre otros dos: un mínimo local del que el algoritmo de Lloyd no puede escapar, con un objetivo casi cuatro veces mayor que el mejor encontrado. Derecha: una ejecución con k-means++, que recupera los cinco grupos. La tabla muestra la dispersión: desde inicios aleatorios simples, dos de las doce ejecuciones terminan en \(J \approx 5200\text{–}5400\), aproximadamente \(3.7\times\) el óptimo \((J \approx 1429)\), mientras que las doce ejecuciones con k-means++ alcanzan el óptimo. k-means++ no garantiza el mínimo global en datos más difíciles, y por eso se combina con reinicios <code>nstart</code>, pero convierte una falla frecuente en una rara a un costo despreciable.</p>

<div class="box warn"><h4>Error frecuente</h4><p>"k-means encontró los clusters" es una afirmación sobre una ejecución desde un inicio. Los mismos datos y el mismo \(k\) pueden producir particiones distintas desde inicios distintos, y el objetivo impreso es la única evidencia interna de cuál es mejor. Ejecuta siempre varias inicializaciones (<code>nstart = 10</code> o más en R), informa siempre el objetivo y nunca interpretes una partición que otro inicio logra mejorar.</p></div>

### 2.6 Elegir k

El número de clusters es una entrada, no una salida, y ningún criterio interno puede determinarlo con certeza: los datos no saben cuántos grupos quiere el analista. Hay tres heurísticas de uso común.

**El método del codo.** Grafica el objetivo \(J^\ast(k)\) frente a \(k\). Siempre disminuye (con \(k = n\) es cero), pero si los datos tienen \(k_0\) grupos cae abruptamente hasta \(k_0\) y lentamente después; el "codo" es el valor en que la ganancia por cada cluster añadido se desploma. El método es visual, a menudo ambiguo, y debe leerse como un rango más que como un valor.

**La silueta.** Para cada punto \(i\), sea \(a(i)\) su distancia media a los demás miembros de su propio cluster y \(b(i)\) la menor distancia media a los miembros de cualquier otro cluster. La silueta del punto es

<div class="eq">\[ s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} \in [-1, 1] \]
<span class="where">que es cercana a 1 cuando el punto está mucho más cerca de su propio cluster que del otro más cercano, cercana a 0 cuando se encuentra entre dos clusters, y negativa cuando probablemente está en el cluster equivocado. La silueta media sobre todos los puntos es un puntaje para la partición completa; se elige el \(k\) que la maximiza.</span></div>

Rousseeuw introdujo la silueta en 1987, y la silueta media se ha convertido en el índice interno más usado.<sup class="c"><a href="#r11">[11]</a></sup> R la calcula con <code>cluster::silhouette</code>; su costo es \(O(n^2)\) porque necesita todas las distancias entre pares.

**El estadístico gap.** Tibshirani, Walther y Hastie propusieron comparar el \(\log J^\ast(k)\) observado con su valor esperado bajo una distribución de referencia <em>nula</em> sin estructura de clusters —datos uniformes sobre la caja que acota la muestra—, estimado a partir de \(B\) conjuntos de datos simulados:<sup class="c"><a href="#r12">[12]</a></sup>

<div class="eq">\[ \mathrm{Gap}(k) = \mathbb{E}^\ast\!\left[\log J^\ast_{\text{null}}(k)\right] - \log J^\ast(k) \]
<span class="where">El \(k\) elegido es el que maximiza el gap (o, con la regla más estricta, el menor \(k\) que queda a menos de un error estándar del máximo). A diferencia del codo y de la silueta, el estadístico gap puede devolver \(k = 1\) —es decir, puede decir "no hay clusters"—, lo que a menudo es la respuesta correcta.</span></div>



In [ ]:
db <- dist(Xb); ks <- 1:9
inertia <- sapply(ks, function(k) if (k == 1) sum(scale(Xb, scale = FALSE)^2)
                  else { set.seed(0); kmeans(Xb, k, nstart = 5)$tot.withinss })
sil <- c(NA, sapply(2:9, function(k) { set.seed(0); mean(silhouette(kmeans(Xb, k, nstart = 5)$cluster, db)[, 3]) }))
gap_statistic <- function(X, ks, B = 20) {               # Gap(k) y su error estándar a partir de B conjuntos de referencia uniformes
  lo <- apply(X, 2, min); hi <- apply(X, 2, max); gaps <- errs <- numeric(0)
  for (k in ks) {
    set.seed(0); logW <- log(kmeans(X, k, nstart = 3)$tot.withinss)
    logWn <- sapply(1:B, function(b) { set.seed(100 + b)
      Xn <- cbind(runif(nrow(X), lo[1], hi[1]), runif(nrow(X), lo[2], hi[2])); log(kmeans(Xn, k, nstart = 3)$tot.withinss) })
    gaps <- c(gaps, mean(logWn) - logW); errs <- c(errs, sd(logWn) * sqrt(1 + 1 / B))
  }
  list(gap = gaps, se = errs)
}
g <- gap_statistic(Xb, ks)
cat("silhouette chooses k =", which.max(sil), "| gap is maximised at k =", which.max(g$gap), "| true number of groups = 5\n")
par(mfrow = c(1, 3), mar = c(4, 4, 2, 1))
plot(ks, inertia, type = "b", pch = 19, col = BLUE, xlab = "k", ylab = "J*(k)", main = "elbow: objective"); abline(v = 5, lty = 3, col = GREY)
plot(2:9, sil[-1], type = "b", pch = 19, col = ORANGE, xlab = "k", ylab = "mean silhouette", main = "silhouette"); abline(v = 5, lty = 3, col = GREY)
plot(ks, g$gap, type = "b", pch = 19, col = GREEN, xlab = "k", ylab = "Gap(k)", main = "gap statistic (+/- s.e.)")
arrows(ks, g$gap - g$se, ks, g$gap + g$se, length = 0.02, angle = 90, code = 3, col = GREEN); abline(v = 5, lty = 3, col = GREY)



<p class="caption"><b>Figura 4.</b> Tres criterios sobre datos generados con cinco grupos (línea punteada). La curva del objetivo tiene su codo en \(k = 5\): la disminución de 4 a 5 es grande, de 5 a 6 pequeña. La silueta media alcanza su máximo en \(k = 5\). El estadístico gap es plano y bajo hasta \(k = 4\), porque con menos de cinco clusters la dispersión intra-cluster sigue siendo casi tan grande como en datos uniformes, y luego salta en \(k = 5\), donde cada grupo por fin obtiene un centroide, de modo que su máximo cae en el valor verdadero. Los tres concuerdan aquí porque los grupos están bien separados y son redondos; en datos reales a menudo discrepan, y el analista informa la discrepancia en lugar del criterio que dio la respuesta deseada. (La regla estricta de "el primero a menos de un error estándar" es frágil cuando los incrementos en \(k\) pequeños son diminutos en relación con el error estándar, como ocurre aquí; el <em>máximo</em> del gap no es ambiguo.)</p>

### 2.7 Modos de falla, demostrados

La Sección 2.1 señaló que la función objetivo prefiere clusters redondos de tamaño y dispersión similares. El bloque de código siguiente construye tres conjuntos de datos que violan una preferencia cada uno y muestra lo que k-means hace con ellos, evaluado con el índice de Rand ajustado (Sección 2.8).



In [ ]:
make_moons <- function(n, noise = 0.06) {                 # medias lunas entrelazadas, un make_moons en R base
  n1 <- n %/% 2; n2 <- n - n1; t1 <- seq(0, pi, length.out = n1); t2 <- seq(0, pi, length.out = n2)
  rbind(cbind(cos(t1), sin(t1)), cbind(1 - cos(t2), 0.5 - sin(t2))) + matrix(rnorm(2 * n, 0, noise), ncol = 2)
}
set.seed(0)
Xa <- rbind(cbind(rnorm(600, 0, 2), rnorm(600, 0, 2)), cbind(rnorm(40, 7, .5), rnorm(40, 5, .5)), cbind(rnorm(40, 7, .5), rnorm(40, -5, .5)))
ya <- rep(1:3, c(600, 40, 40))
Xb_moons <- make_moons(400, 0.06); yb_moons <- rep(1:2, c(200, 200))
base_c <- rbind(cbind(rnorm(150, 0, 3), rnorm(150, 0, .6)), cbind(rnorm(150, 0, 3), rnorm(150, 3, .6)))  # los grupos difieren a lo largo de la característica 2
yc <- rep(1:2, c(150, 150)); Xc <- base_c %*% diag(c(50, 1))                                             # característica 1: escala grande e irrelevante
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1))
for (d in list(list("(a) unequal sizes", Xa, ya, 3), list("(b) non-spherical", Xb_moons, yb_moons, 2), list("(c) unscaled features", Xc, yc, 2))) {
  set.seed(0); km <- kmeans(d[[2]], d[[4]], nstart = 10)
  plot(d[[2]], col = PALETTE[km$cluster], pch = 19, cex = 0.5, xlab = "", ylab = "",
       main = sprintf("%s\nARI = %.2f", d[[1]], adjusted_rand(d[[3]], km$cluster))); points(km$centers, pch = 4, lwd = 2, cex = 1.6)
}
cat("(c) after standardising both features: ARI =", round(adjusted_rand(yc, kmeans(scale(Xc), 2, nstart = 10)$cluster), 2), "\n")



<p class="caption"><b>Figura 5.</b> Tres fallas. (a) Un grupo grande y difuso y dos pequeños y compactos: k-means con \(k = 3\) sitúa un centroide en cada grupo pequeño pero le asigna también una amplia franja del grupo grande, porque la frontera entre dos centroides es la mediatriz del segmento que los une, que cae muy adentro del grupo difuso; el ARI bajo contabiliza los puntos mal asignados. (b) Dos medias lunas: k-means solo puede trazar fronteras rectas entre centroides, así que corta cada luna por la mitad (ARI cercano a 0.2). (c) Dos grupos redondos separados a lo largo de la característica 2, pero la característica 1 se mide en una escala cincuenta veces mayor y es irrelevante para el agrupamiento: la distancia euclidiana queda dominada por la característica 1 y la división se hace a lo largo de su ruido, ignorando la separación real (ARI cercano a 0); estandarizar ambas características restablece la partición correcta (ARI impreso \(\approx\) 0.99).</p>

Cada falla tiene un remedio en otra parte de la unidad: las dispersiones desiguales se manejan con mezclas gaussianas, que ajustan una covarianza por cluster (Sección 6); las formas no convexas, con agrupamiento jerárquico de enlace simple o agrupamiento espectral (Secciones 3 y 6); las características sin escalar, con estandarización, que es la razón por la que la Sección 1.3 estandarizó los datos de cáncer de mama antes que cualquier otra cosa.

### 2.8 k-means frente a las etiquetas ocultas en los datos de cáncer de mama

Por fin se agrupa el ejemplo conductor. Se ejecuta k-means con \(k = 2\) sobre las nueve características estandarizadas, y la partición se compara con el resultado de biopsia que el algoritmo nunca vio. La comparación usa una <strong>tabla de contingencia</strong> —cluster frente a etiqueta— y el <strong>índice de Rand ajustado</strong> (ARI, adjusted Rand index) de Hubert y Arabie,<sup class="c"><a href="#r13">[13]</a></sup> que cuenta los pares de casos que ambas particiones concuerdan en poner juntos o separados, corregido de modo que una partición aleatoria obtenga 0 y una coincidencia perfecta obtenga 1. El ARI es invariante a los nombres de los clusters, lo cual es esencial: k-means llama a sus grupos 1 y 2 en un orden arbitrario que no tiene nada que ver con "maligno" y "benigno".



In [ ]:
set.seed(0); km2 <- kmeans(X_std, centers = 2, nstart = 10)
print(table(`k-means cluster` = km2$cluster, biopsy = y_hidden))
agree <- max(mean(km2$cluster == as.integer(y_hidden)), mean(km2$cluster != as.integer(y_hidden)))
cat(sprintf("\nadjusted Rand index = %.3f\n", adjusted_rand(y_hidden, km2$cluster)))
cat(sprintf("agreement with the biopsy under the best relabelling = %.1f%%  (NOT an accuracy: no label was used to fit)\n", 100 * agree))
set.seed(0); km2_raw <- kmeans(X_raw, centers = 2, nstart = 10)
cat(sprintf("the same with the UNstandardised features: ARI = %.3f, cluster sizes = %s\n",
            adjusted_rand(y_hidden, km2_raw$cluster), paste(as.vector(table(km2_raw$cluster)), collapse = ", ")))



Con las características estandarizadas, un cluster contiene 434 de los 444 casos benignos y el otro contiene 220 de los 239 malignos; el índice de Rand ajustado es de aproximadamente 0.84 y, reetiquetando a posteriori los clusters como diagnósticos, alrededor del 96 % de los casos caen del lado "correcto". Aquí está el contraste prometido con la edición Python: con las características sin estandarizar, el ARI es <em>todavía</em> de aproximadamente 0.85 y la partición es solo ligeramente distinta. La razón es estructural: las nueve características de mlbench son todas puntuaciones citológicas en la misma escala 1–10, de modo que ninguna característica por sí sola puede dominar la distancia como <em>worst area</em>, en µm², dominaba los datos de 30 características. La estandarización es una buena práctica y no cuesta nada, pero rescata el resultado solo cuando las características en bruto están en escalas inconmensurables, que es exactamente la situación de la Figura 5c y <em>no</em> la situación de aquí. La afirmación honesta es que, para este conjunto de datos en particular, el paso apenas importa, y saber por qué vale más que el reflejo de escalar siempre.

<div class="box warn"><h4>La corrección de la Unidad 2, reiterada</h4><p>La concordancia del 96 % no es una exactitud y k-means no es un clasificador. Un clasificador aprende una correspondencia entre las características y <em>la etiqueta</em>; k-means aprendió una partición del espacio de características que minimiza la dispersión intra-cluster y que resulta —porque el tamaño y la forma nucleares son la fuente dominante de variación en estos datos, y también la base del diagnóstico— alinearse con la etiqueta. De ello se siguen tres consecuencias. (i) La alineación no puede conocerse sin etiquetas, de modo que en un estudio genuinamente sin etiquetas no puede afirmarse. (ii) La partición no mejorará con más casos, porque no entra en ella ninguna información sobre la etiqueta. (iii) Un clasificador supervisado con los mismos datos alcanza una concordancia mayor (Unidades 3–4) porque optimiza lo correcto. El uso correcto de este resultado es el de la Sección 8: cuando las etiquetas <em>no</em> existen, el agrupamiento es la forma de descubrir que los casos se reparten en grupos que vale la pena nombrar.</p></div>

## 3. Agrupamiento jerárquico

### 3.1 El algoritmo aglomerativo

k-means necesita \(k\) de antemano y devuelve una sola partición. El agrupamiento jerárquico devuelve <em>todas</em> las particiones a la vez, anidadas unas dentro de otras, en forma de árbol —el <strong>dendrograma</strong>— y \(k\) se elige después, cortando el árbol a cierta altura. El algoritmo <strong>aglomerativo</strong> (de abajo hacia arriba) es:

<ol>
<li>Comenzar con \(n\) clusters, uno por punto, y la matriz \(n \times n\) de distancias entre pares.</li>
<li>Encontrar los dos clusters a menor distancia y <strong>fusionarlos</strong>. Registrar la fusión y su distancia (la <em>altura</em> de la fusión).</li>
<li>Actualizar la matriz de distancias: calcular la distancia entre el nuevo cluster y cada cluster restante usando una regla de <strong>enlace</strong> (linkage).</li>
<li>Repetir los pasos 2–3 hasta que quede un solo cluster. Las \(n - 1\) fusiones registradas constituyen el dendrograma.</li>
</ol>

Todo lo que distingue un método jerárquico de otro está en el paso 3: la definición de la distancia entre dos <em>clusters</em>, dadas las distancias entre sus <em>puntos</em>. La implementación ingenua cuesta \(O(n^3)\) en tiempo y \(O(n^2)\) en memoria para la matriz de distancias, lo que limita el método a algunas decenas de miles de casos; para \(n\) mayores se agrupa una muestra o los centroides de un k-means preliminar. En R el algoritmo es <code>hclust</code>, que recibe un objeto <code>dist</code>.

### 3.2 Distancias entre clusters: las reglas de enlace

Para clusters \(A\) y \(B\) con \(|A|\) y \(|B|\) puntos y distancias entre pares de puntos \(d(\mathbf{a}, \mathbf{b})\):

| Enlace (`method=` de R) | \(D(A, B)\) | Carácter |
|---|---|---|
| **Simple** `"single"` | \(\min_{\mathbf{a} \in A,\, \mathbf{b} \in B} d(\mathbf{a}, \mathbf{b})\) | Fusiona clusters que se tocan en cualquier punto; sigue formas alargadas; propenso al <em>encadenamiento</em> (chaining, Sección 3.5) |
| **Completo** `"complete"` | \(\max_{\mathbf{a} \in A,\, \mathbf{b} \in B} d(\mathbf{a}, \mathbf{b})\) | Fusiona solo clusters cuyos puntos más lejanos están cerca; clusters compactos y redondos; sensible a valores atípicos |
| **Promedio** `"average"` (UPGMA) | \(\dfrac{1}{|A|\,|B|} \sum_{\mathbf{a}} \sum_{\mathbf{b}} d(\mathbf{a}, \mathbf{b})\) | Un compromiso; el método de los árboles filogenéticos de BGM324 y de los mapas de calor de expresión génica de la Sección 3.6 |
| **Ward** `"ward.D2"` | \(\sqrt{\dfrac{2\,|A|\,|B|}{|A| + |B|}}\; \lVert \bar{\mathbf{a}} - \bar{\mathbf{b}} \rVert_2\) | Fusiona el par cuya unión aumenta menos la suma de cuadrados intra-cluster; la contraparte jerárquica de k-means |

Fusionar \(A\) y \(B\) aumenta la suma de cuadrados intra-cluster en \(\frac{|A||B|}{|A|+|B|} \lVert \bar{\mathbf{a}} - \bar{\mathbf{b}} \rVert^2\) (con centroides \(\bar{\mathbf{a}}, \bar{\mathbf{b}}\)); el método <code>"ward.D2"</code> de R informa la raíz cuadrada del doble de esa cantidad para que, en el caso de dos puntos individuales, sea igual a su distancia euclidiana. El artículo de Ward de 1963 define el método como la fusión que minimiza la pérdida de una función objetivo, y la suma de cuadrados del error es el caso que se volvió estándar.<sup class="c"><a href="#r15">[15]</a></sup> Las cuatro reglas son casos particulares de la recurrencia de Lance–Williams,<sup class="c"><a href="#r14">[14]</a></sup> que expresa la distancia entre un cluster fusionado \(A \cup B\) y cualquier otro cluster \(C\) como una combinación lineal de \(D(A, C)\), \(D(B, C)\) y \(D(A, B)\), de modo que la matriz de distancias puede actualizarse sin volver a los puntos. (El histórico <code>"ward.D"</code> de R omite la raíz cuadrada y no debe usarse; <code>"ward.D2"</code> es la implementación correcta del criterio de Ward.)

### 3.3 Ejemplo resuelto a mano: las tres primeras fusiones

Ocho puntos en dos dimensiones, con coordenadas a un decimal para que cada distancia pueda verificarse en papel. Tres pares compactos están muy separados entre sí, con otros dos puntos aislados. El bloque de código imprime los puntos y las seis menores distancias entre pares; a continuación viene el seguimiento a mano.



In [ ]:
H <- matrix(c(1.0,1.0, 1.3,1.0,  5.0,5.0, 5.6,5.0,  1.0,5.0, 1.0,5.8,  3.0,3.0, 7.0,1.0),
            ncol = 2, byrow = TRUE, dimnames = list(paste0("x", 1:8), c("x1", "x2")))
Dm <- as.matrix(dist(H)); Dm[lower.tri(Dm, diag = TRUE)] <- NA
pairs <- which(!is.na(Dm), arr.ind = TRUE)
ord <- order(Dm[!is.na(Dm)]); six <- pairs[order(Dm[pairs]), ][1:6, ]
cat("the eight points:\n"); print(H)
cat("\nsix smallest pairwise distances:\n")
for (r in 1:6) cat(sprintf("  x%d - x%d : %.3f\n", six[r, 1], six[r, 2], Dm[six[r, 1], six[r, 2]]))



**Fusión 1.** La menor distancia es \(d(\mathbf{x}_1, \mathbf{x}_2) = \sqrt{0.3^2 + 0^2} = 0.3\). El cluster \(\{1, 2\}\) se forma a la altura 0.3.

**Fusión 2.** La siguiente menor es \(d(\mathbf{x}_3, \mathbf{x}_4) = \sqrt{0.6^2} = 0.6\). Como \(\mathbf{x}_3\) y \(\mathbf{x}_4\) están aislados de todo lo demás, ninguna regla de enlace puede hacer que el cluster \(\{1, 2\}\) quede más cerca de algún punto que 0.6, de modo que esta fusión es \(\{3, 4\}\) a la altura 0.6 para <em>todo</em> enlace.

**Fusión 3.** Por el mismo razonamiento, \(d(\mathbf{x}_5, \mathbf{x}_6) = \sqrt{0.8^2} = 0.8\): la tercera fusión es \(\{5, 6\}\) a la altura 0.8, de nuevo para todo enlace.

**Dónde divergen los enlaces.** Las reglas coinciden en estas tres fusiones solo porque los pares están aislados. Considera la distancia actualizada entre el cluster \(\{1, 2\}\) y el punto \(\mathbf{x}_7 = (3, 3)\): \(d(\mathbf{x}_1, \mathbf{x}_7) = \sqrt{2^2 + 2^2} = 2.828\) y \(d(\mathbf{x}_2, \mathbf{x}_7) = \sqrt{1.7^2 + 2^2} = 2.625\). El enlace simple registra \(\min = 2.625\), el enlace completo \(\max = 2.828\), el enlace promedio \((2.625 + 2.828)/2 = 2.727\), y Ward, con el centroide de \(\{1, 2\}\) en \((1.15, 1)\), registra \(\sqrt{\tfrac{2 \cdot 2 \cdot 1}{3}}\, \lVert (1.15,1) - (3,3) \rVert = 1.155 \times 2.724 = 3.146\). A medida que se acumulan las fusiones, estas diferencias se multiplican y los cuatro dendrogramas se convierten en árboles distintos. El bloque de código confirma las tres alturas de fusión y las cuatro distancias actualizadas.



In [ ]:
hc_methods <- c("single", "complete", "average", "ward.D2")
first3 <- sapply(hc_methods, function(m) round(hclust(dist(H), m)$height[1:3], 3))
cat("first three merge heights by linkage (rows = merge 1,2,3):\n"); print(first3)
A <- H[c(1, 2), ]; p7 <- H[7, ]; dp <- sqrt(rowSums(sweep(A, 2, p7)^2))
cat(sprintf("\nD({1,2}, x7): single = %.3f, complete = %.3f, average = %.3f, ward = %.3f\n",
            min(dp), max(dp), mean(dp), sqrt(2 * 2 * 1 / 3) * sqrt(sum((colMeans(A) - p7)^2))))
par(mfrow = c(1, 2), mar = c(2, 4, 2, 1))
plot(hclust(dist(H), "average"), main = "average linkage", xlab = "", sub = "", ylab = "merge height")
plot(hclust(dist(H), "ward.D2"), main = "Ward linkage", xlab = "", sub = "", ylab = "merge height")



<p class="caption"><b>Figura 6.</b> Dendrogramas de los ocho puntos con enlace promedio (izquierda) y enlace de Ward (derecha). Leyendo desde abajo, las tres uniones más bajas son las tres fusiones seguidas a mano —\(\{1,2\}\) a 0.3, \(\{3,4\}\) a 0.6, \(\{5,6\}\) a 0.8—, idénticas en ambos árboles; más arriba, los árboles difieren en el orden y la altura de las fusiones posteriores, y las alturas de Ward crecen más rápido porque su criterio penaliza la fusión de clusters grandes. Los valores confirmados de \(D(\{1,2\}, \mathbf{x}_7)\) —2.625, 2.828, 2.727 y 3.146 para enlace simple, completo, promedio y de Ward— coinciden exactamente con el cálculo a mano.</p>

### 3.4 Cortar el árbol y la correlación cofenética

Un dendrograma se convierte en una partición cuando se corta horizontalmente: cada rama que cruza la altura de corte es un cluster (<code>cutree(hc, k)</code> corta para obtener \(k\) clusters). La altura "correcta" suele leerse en el dendrograma como el tramo vertical más largo sin fusiones: un gran salto entre alturas de fusión sucesivas significa que los clusters que quedan por debajo son mucho más parecidos internamente que entre sí.

Un dendrograma también implica una distancia entre dos puntos cualesquiera: la altura a la que se unen por primera vez, la <strong>distancia cofenética</strong>. El <strong>coeficiente de correlación cofenética</strong> es la correlación de Pearson entre las distancias cofenéticas y las distancias originales sobre todos los pares;<sup class="c"><a href="#r16">[16]</a></sup> mide con qué fidelidad el árbol representa la matriz de distancias y es la forma estándar de comparar reglas de enlace sobre los mismos datos (valores superiores a aproximadamente 0.75 indican una representación fiel). R proporciona <code>cophenetic(hc)</code>. El bloque de código toma un subconjunto de 20 casos de los datos de cáncer de mama con dos características, corta cada árbol en dos clusters, los compara con las etiquetas ocultas e informa la correlación cofenética de cada enlace.



In [ ]:
set.seed(0); si <- sample(nrow(X_raw), 20)
sub <- X_raw[si, c("Cell.size", "Cl.thickness")]; sub_lab <- y_hidden[si]; dsub <- dist(sub)
tab <- data.frame(t(sapply(hc_methods, function(m) {
  hc <- hclust(dsub, m); cut <- cutree(hc, 2)
  c(coph = round(cor(cophenetic(hc), dsub), 3),
    ARI  = round(adjusted_rand(sub_lab, cut), 3),
    size1 = min(table(cut)), size2 = max(table(cut)),
    max_gap = round(max(diff(hc$height)), 3))
})))
print(tab)



La tabla deja tres enseñanzas. Primera: aquí la correlación cofenética es máxima para el enlace simple o el promedio (las cuatro son cercanas, 0.85–0.87, porque el subconjunto de 20 puntos es pequeño y sus distancias modestas); el enlace promedio es, por construcción, la regla que en general mejor preserva las distancias originales, y por eso es la opción por defecto en filogenética y en los mapas de calor de la Sección 3.6. Segunda: cortar los árboles en \(k = 2\) da particiones de distinto equilibrio y distinta concordancia con el diagnóstico; el enlace simple, en particular, tiende a desprender uno o dos casos periféricos y dejar el resto junto, la versión a pequeña escala del efecto de encadenamiento que se muestra a continuación. Tercera: la columna "max_gap" es una lectura automática burda del dendrograma; es una heurística, no una prueba.

### 3.5 El efecto de encadenamiento del enlace simple

El enlace simple fusiona dos clusters en cuanto <em>cualquier</em> par de sus puntos está cerca. Una delgada línea de puntos entre dos grupos —unos pocos pacientes intermedios, una transición ruidosa, un puente— basta, por lo tanto, para unir los grupos, eslabón por eslabón, antes de que los propios grupos estén completos. Los clusters resultantes son cadenas largas más que conjuntos compactos. La misma propiedad es una virtud cuando los clusters son realmente alargados o curvos (las medias lunas de la Figura 5b), de modo que el efecto es un compromiso, no un error.



In [ ]:
set.seed(0)
g1 <- cbind(rnorm(100, 0, .5), rnorm(100, 0, .5)); g2 <- cbind(rnorm(100, 6, .5), rnorm(100, 0, .5))
bridge <- cbind(seq(0.8, 5.2, length.out = 12), rnorm(12, 0, .05))
Xc2 <- rbind(g1, g2, bridge)
par(mfrow = c(1, 3), mar = c(2, 2, 3, 1))
for (m in c("single", "average", "ward.D2")) {
  lab <- cutree(hclust(dist(Xc2), m), 2)
  plot(Xc2, col = PALETTE[lab], pch = 19, cex = 0.5, asp = 1, xlab = "", ylab = "",
       main = sprintf("%s, k = 2\nsizes %s", m, paste(as.vector(table(lab)), collapse = ", ")))
  cat(sprintf("%-8s on the half-moons (k = 2): ARI = %.2f\n", m,
              adjusted_rand(yb_moons, cutree(hclust(dist(Xb_moons), m), 2))))
}



<p class="caption"><b>Figura 7.</b> Doscientos puntos en dos grupos compactos, más doce puntos que forman un puente. El enlace simple (izquierda) sigue el puente y une ambos grupos en un solo cluster, dejando un único punto atípico como "segundo cluster". Los enlaces promedio y de Ward (centro, derecha) cortan el puente y recuperan los dos grupos. La salida impresa muestra la otra cara del compromiso: en las medias lunas de la Figura 5b, el enlace simple es el único de los tres que recupera las dos formas curvas (ARI \(\approx 1\)), porque el encadenamiento es exactamente lo que se necesita para seguir una curva, mientras que los enlaces promedio y de Ward obtienen alrededor de 0.4 y 0.5.</p>

**Métodos divisivos.** La estrategia opuesta —comenzar con un solo cluster que lo contiene todo y dividirlo recursivamente— es el agrupamiento jerárquico <strong>divisivo</strong>. Rara vez se usa en su forma exacta porque elegir la mejor división de un cluster es combinatorio (hay \(2^{n-1} - 1\) formas de dividir \(n\) puntos en dos); el algoritmo práctico DIANA<sup class="c"><a href="#r8">[8]</a></sup> (disponible como <code>cluster::diana</code>) divide sacando uno a uno los puntos más disímiles, y el <em>bisecting k-means</em> (k-means bisectante) divide en cada paso el cluster más grande o menos compacto con \(k = 2\). Los métodos divisivos aciertan primero en la parte superior del árbol —la estructura gruesa—, mientras que los métodos aglomerativos, una vez comprometidos con las fusiones tempranas, no pueden deshacerlas.

### 3.6 Aplicación: mapas de calor de expresión génica

En 1998, Eisen, Spellman, Brown y Botstein aplicaron agrupamiento jerárquico de enlace promedio, con una distancia basada en la correlación, a la expresión de miles de genes de levadura a lo largo de series temporales, reordenaron las filas y columnas de la matriz de expresión según los dendrogramas y mostraron el resultado como una imagen en color: el "mapa de calor agrupado" (clustered heatmap) que desde entonces ha sido la visualización estándar de los datos de expresión.<sup class="c"><a href="#r17">[17]</a></sup> La idea es sencilla: los genes con perfiles de expresión similares terminan adyacentes, de modo que aparecen bloques de color donde grupos de genes se comportan de forma parecida en grupos de condiciones, y tanto los clusters de genes como los de muestras pueden leerse en los márgenes. La edición Python construyó su miniatura con la tabla de química de vinos de scikit-learn<sup class="c"><a href="#r18">[18]</a></sup> como sustituto de una matriz muestras × genes; R no trae incorporado un conjunto de datos así, de modo que el bloque de código <em>simula</em> una pequeña matriz al estilo de los datos de expresión, con tres grupos de muestras y bloques correlacionados de características —lo que en realidad se parece más a la aplicación de Eisen que los datos de vinos— y agrupa ambos ejes con enlace promedio sobre la distancia de correlación, exactamente como lo hizo Eisen. La función <code>heatmap</code> de R realiza el agrupamiento y el reordenamiento en una sola llamada.



In [ ]:
set.seed(0)
# matriz de expresión simulada: 60 muestras en 3 grupos x 18 características en 3 bloques que covarían
make_expr <- function() {
  grp <- rep(1:3, each = 20); block_means <- matrix(c(2,-1,-1, -1,2,-1, -1,-1,2), 3, 3, byrow = TRUE)
  M <- matrix(0, 60, 18)
  for (g in 1:3) for (bk in 1:3) {
    cols <- ((bk - 1) * 6 + 1):(bk * 6)
    M[grp == g, cols] <- rnorm(20 * 6, block_means[g, bk], 0.7)
  }
  list(M = scale(M), grp = grp)
}
ex <- make_expr(); colnames(ex$M) <- paste0("g", 1:18); rownames(ex$M) <- paste0("s", 1:60)
cc_dist <- function(x) as.dist(1 - cor(t(x)))                        # distancia de correlación entre filas
side <- c(BLUE, ORANGE, GREEN)[ex$grp]
heatmap(ex$M, distfun = cc_dist, hclustfun = function(d) hclust(d, "average"),
        RowSideColors = side, col = colorRampPalette(c(BLUE, "white", ORANGE))(40),
        scale = "none", margins = c(4, 4), xlab = "features (genes)", ylab = "samples")
cut3 <- cutree(hclust(cc_dist(ex$M), "average"), 3)
cat("three sample clusters vs. the hidden group: ARI =", round(adjusted_rand(ex$grp, cut3), 2),
    "; cluster sizes", paste(as.vector(table(cut3)), collapse = ", "), "\n")



<p class="caption"><b>Figura 8.</b> Un mapa de calor agrupado al estilo de Eisen et al. Cada fila es una muestra simulada y cada columna una característica estandarizada (naranja por encima de la media, azul por debajo); filas y columnas se han reordenado según dendrogramas de enlace promedio sobre la distancia de correlación, dibujados en los márgenes. Aparecen bloques de color, uno por cada grupo de muestras que comparte un bloque alto de características, porque la simulación construyó exactamente esa estructura. La barra de color de la izquierda muestra el grupo oculto de cada muestra y revela que los clusters de muestras coinciden con los tres grupos (ARI impreso arriba). Lo que la figura <em>no</em> muestra es la matriz en bruto en su orden original, que sería ruido para el ojo; la estructura la produce el reordenamiento.</p>

<div class="box bridge"><h4>Puente con bioinformática</h4><p>El árbol UPGMA de la Unidad 1 de BGM324, construido a partir de una matriz de distancias entre pares de secuencias, <em>es</em> agrupamiento aglomerativo de enlace promedio: las hojas son secuencias, las alturas de fusión son distancias evolutivas y las distancias cofenéticas son las longitudes de rama. El mapa de calor agrupado es el mismo algoritmo aplicado dos veces, a las filas y a las columnas de una matriz de expresión, y la Sección 8 describe cómo definió los subtipos moleculares del cáncer de mama.</p></div>

## 4. Análisis de componentes principales

### 4.1 Dos visiones de una misma idea

El análisis de componentes principales (PCA) reemplaza \(p\) características correlacionadas por \(q \le p\) características nuevas —las <strong>componentes principales</strong>— que son combinaciones lineales de las originales, no correlacionadas entre sí y ordenadas según la cantidad de varianza que contienen. Dos formulaciones conducen a las mismas componentes.

**Maximización de la varianza.** Encontrar el vector unitario \(\mathbf{w}_1\) tal que la proyección de los datos (centrados) sobre él, \(z_i = \mathbf{w}_1^\top \mathbf{x}_i\), tenga la mayor varianza posible; luego encontrar \(\mathbf{w}_2 \perp \mathbf{w}_1\) con la mayor varianza restante, y así sucesivamente. Esta es la formulación de Hotelling de 1933,<sup class="c"><a href="#r19">[19]</a></sup> y aquella de la que el método toma su nombre.

**Mínimo error de reconstrucción.** Encontrar el subespacio de dimensión \(q\) (una recta para \(q = 1\), un plano para \(q = 2\)) tal que la suma de las distancias perpendiculares al cuadrado desde los puntos al subespacio sea mínima. Esta es la formulación de Pearson de 1901 —"rectas y planos de mejor ajuste a sistemas de puntos en el espacio"<sup class="c"><a href="#r20">[20]</a></sup>— y dice que PCA es la mejor <em>compresión lineal</em> posible de los datos: las \(q\) coordenadas de cada punto en ese subespacio son el resumen a partir del cual el punto puede reconstruirse con el menor error cuadrático total.

Ambas coinciden porque, para datos centrados, la distancia al cuadrado de un punto al origen es igual a la longitud al cuadrado de su proyección sobre el subespacio más la distancia perpendicular al cuadrado a él; la primera suma es fija, de modo que maximizar la varianza proyectada y minimizar el residuo son el mismo problema. Ambas conducen a la matriz de covarianza.

### 4.2 Covarianza, vectores propios, valores propios: ejemplo resuelto a mano

Sea \(\mathbf{X}\) la matriz de datos \(n \times p\) a la que se han restado las medias de las columnas. Su <strong>matriz de covarianza muestral</strong> es

<div class="eq">\[ \mathbf{S} = \frac{1}{n-1}\,\mathbf{X}^\top \mathbf{X}, \qquad S_{jl} = \frac{1}{n-1}\sum_{i=1}^{n} x_{ij}\,x_{il} \]
<span class="where">una matriz simétrica \(p \times p\) cuya diagonal contiene las varianzas de las características y cuyos elementos fuera de la diagonal contienen sus covarianzas. La varianza de una proyección \(\mathbf{w}^\top \mathbf{x}\) es \(\mathbf{w}^\top \mathbf{S}\, \mathbf{w}\), y maximizarla sujeta a \(\lVert \mathbf{w} \rVert = 1\) (mediante un multiplicador de Lagrange) da \(\mathbf{S}\mathbf{w} = \lambda \mathbf{w}\): las direcciones de varianza máxima son los <strong>vectores propios</strong> (eigenvectores) de \(\mathbf{S}\), y la varianza a lo largo de cada uno es su <strong>valor propio</strong> (eigenvalor) \(\lambda\). Como \(\mathbf{S}\) es simétrica, sus vectores propios son ortogonales y sus valores propios reales y no negativos. Ordenados de forma decreciente, \(\lambda_1 \ge \dots \ge \lambda_p\), los vectores propios son las componentes principales, y \(\lambda_j / \sum_l \lambda_l\) es la <strong>proporción de varianza explicada</strong> por la componente \(j\).</span></div>

**Ejemplo resuelto.** Cinco puntos en dos dimensiones: \((2,1), (3,3), (4,2), (5,4), (6,5)\).

**Paso 1 — centrar.** Medias \(\bar{x}_1 = 4\), \(\bar{x}_2 = 3\). Datos centrados: \((-2, -2), (-1, 0), (0, -1), (1, 1), (2, 2)\).

**Paso 2 — matriz de covarianza.** Con \(n - 1 = 4\): \(S_{11} = (4 + 1 + 0 + 1 + 4)/4 = 2.5\); \(S_{22} = 2.5\); \(S_{12} = (4 + 0 + 0 + 1 + 4)/4 = 2.25\). Por lo tanto, \(\mathbf{S} = \begin{pmatrix} 2.5 & 2.25 \\ 2.25 & 2.5 \end{pmatrix}\).

**Paso 3 — valores propios.** \(\det(\mathbf{S} - \lambda \mathbf{I}) = (2.5 - \lambda)^2 - 2.25^2 = 0\), es decir, \(2.5 - \lambda = \pm 2.25\): \(\lambda_1 = 4.75\), \(\lambda_2 = 0.25\). La traza se conserva: \(4.75 + 0.25 = 5 = S_{11} + S_{22}\). Proporción explicada por PC1: \(4.75 / 5 = 0.95\).

**Paso 4 — vectores propios.** Para \(\lambda_1 = 4.75\): \(-2.25 w_1 + 2.25 w_2 = 0\), de modo que \(\mathbf{w}_1 = (1, 1)/\sqrt{2} \approx (0.707, 0.707)\). Para \(\lambda_2 = 0.25\): \(\mathbf{w}_2 = (1, -1)/\sqrt{2} \approx (0.707, -0.707)\), ortogonal a \(\mathbf{w}_1\).

**Paso 5 — proyectar.** La puntuación (score) en PC1 de cada punto centrado es \(z_{i1} = (x_{i1} + x_{i2})/\sqrt{2} \approx -2.83, -0.71, -0.71, 1.41, 2.83\); su varianza es \(4.75 = \lambda_1\). Las puntuaciones en PC2 son \(0, -0.71, 0.71, 0, 0\), cuyos cuadrados suman \(1.0\): el error de reconstrucción cuadrático total si los datos se comprimen a una dimensión. El bloque de código confirma cada número.



In [ ]:
Q  <- matrix(c(2,1, 3,3, 4,2, 5,4, 6,5), ncol = 2, byrow = TRUE)
Qc <- scale(Q, center = TRUE, scale = FALSE)         # paso 1: centrar
S  <- t(Qc) %*% Qc / (nrow(Q) - 1)                   # paso 2: matriz de covarianza
eg <- eigen(S)                                        # pasos 3-4: descomposición en valores propios (decreciente)
scores <- Qc %*% eg$vectors                           # paso 5: proyectar
cat("covariance matrix:\n"); print(S)
cat("eigenvalues:", round(eg$values, 4), "| proportion explained:", round(eg$values / sum(eg$values), 3), "\n")
cat("eigenvectors (columns):\n"); print(round(eg$vectors, 4))
cat("PC1 scores:", round(scores[, 1], 3), "| var:", round(var(scores[, 1]), 3), "\n")
cat("sum of squared PC2 scores (1-D reconstruction error):", round(sum(scores[, 2]^2), 3), "\n")
pr <- prcomp(Q)
cat("prcomp eigenvalues (sdev^2):", round(pr$sdev^2, 4), "| rotation PC1:", round(pr$rotation[, 1], 4), "\n")



<code>eigen</code> y <code>prcomp</code> reproducen el cálculo a mano: valores propios 4.75 y 0.25, las proporciones 0.95 y 0.05, vectores propios \(\pm(0.707, 0.707)\) y \(\pm(0.707, -0.707)\), y las puntuaciones. Los signos de los vectores propios pueden diferir entre la solución a mano, <code>eigen</code> y <code>prcomp</code> —un vector propio multiplicado por \(-1\) sigue siendo un vector propio—, de modo que el signo de una componente no tiene significado y las puntuaciones a lo largo de ella pueden aparecer invertidas de una ejecución o de una función a otra. Esta es la primera de las dos ambigüedades de signo/escala (la segunda pertenece a ICA, Sección 5.3).



In [ ]:
w1 <- eg$vectors[, 1]; proj <- sweep(outer(scores[, 1], w1), 2, colMeans(Q), "+")  # reconstruir solo a partir de PC1
par(mar = c(4, 4, 2, 1))
plot(Q, pch = 19, cex = 1.4, asp = 1, xlim = c(0.5, 7.5), ylim = c(-0.5, 6.5),
     xlab = expression(x[1]), ylab = expression(x[2]), main = "PCA of five points: axis of maximal variance = line of closest fit")
m <- colMeans(Q)
for (sgn in c(-1, 1)) { segments(m[1], m[2], m[1] + sgn * 2.6 * w1[1], m[2] + sgn * 2.6 * w1[2], col = BLUE, lwd = 2)
                        segments(m[1], m[2], m[1] + sgn * 1.2 * eg$vectors[1,2], m[2] + sgn * 1.2 * eg$vectors[2,2], col = ORANGE, lwd = 2) }
for (i in 1:5) segments(Q[i, 1], Q[i, 2], proj[i, 1], proj[i, 2], col = GREY, lty = 3)
points(proj, pch = 19, col = BLUE, cex = 0.9); points(m[1], m[2], pch = 3, cex = 2, lwd = 2)
legend("topleft", c("data", "reconstruction from PC1", "PC1 (95%)", "PC2 (5%)"),
       col = c("black", BLUE, BLUE, ORANGE), pch = c(19, 19, NA, NA), lty = c(NA, NA, 1, 1), lwd = 2, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 9.</b> Los cinco puntos (negro), su media (cruz), el primer eje principal (azul, a lo largo de \((1,1)/\sqrt{2}\)) y el segundo (naranja, perpendicular). Los segmentos punteados son las distancias perpendiculares de cada punto al primer eje; sus longitudes al cuadrado suman 1.0, el error de reconstrucción del Paso 5. Los puntos azules sobre el eje son los puntos tal como se reconstruirían a partir de un único número cada uno, su puntuación en PC1. La visión de Hotelling (varianza máxima a lo largo del eje azul) y la de Pearson (mínimas distancias punteadas al cuadrado) describen la misma recta.</p>

### 4.3 Varianza explicada y gráfico de sedimentación: con y sin escalado

Con nueve características, la matriz de covarianza es de \(9 \times 9\) y tiene nueve valores propios. El <strong>gráfico de sedimentación</strong> (scree plot) los dibuja (o las proporciones de varianza) en orden decreciente; el número de componentes que se conservan se lee ya sea en un codo del gráfico o en el punto en que la proporción acumulada cruza un umbral como 80 % o 95 %. El bloque de código calcula los valores propios para las características en bruto y para las estandarizadas.



In [ ]:
pca_raw <- prcomp(X_raw, center = TRUE, scale. = FALSE)
pca_std <- prcomp(X_raw, center = TRUE, scale. = TRUE)
vr_raw <- pca_raw$sdev^2 / sum(pca_raw$sdev^2); vr_std <- pca_std$sdev^2 / sum(pca_std$sdev^2)
cat(sprintf("RAW:          PC1 %.1f%%, PC2 %.1f%%; PC1 dominated by '%s'\n",
            100 * vr_raw[1], 100 * vr_raw[2], names(which.max(abs(pca_raw$rotation[, 1])))))
cat(sprintf("STANDARDISED: PC1 %.1f%%, PC2 %.1f%%, PC3 %.1f%%; first two %.1f%%; %d components reach 95%%\n",
            100 * vr_std[1], 100 * vr_std[2], 100 * vr_std[3], 100 * sum(vr_std[1:2]), which(cumsum(vr_std) >= 0.95)[1]))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
for (z in list(list(vr_raw, "raw features"), list(vr_std, "standardised features"))) {
  barplot(z[[1]], names.arg = 1:9, col = BLUE, ylim = c(0, 1), xlab = "principal component", ylab = "proportion of variance", main = z[[2]])
  lines(seq(0.7, by = 1.2, length.out = 9), cumsum(z[[1]]), type = "b", pch = 19, col = ORANGE); abline(h = 0.95, lty = 3, col = GREY)
}



<p class="caption"><b>Figura 10.</b> Gráficos de sedimentación. Como las nueve características de mlbench ya comparten la escala 1–10, los panoramas en bruto y estandarizado son similares: PC1 contiene alrededor del 69 % de la varianza en bruto y el 66 % estandarizada, y siete componentes llegan al 95 % en ambos casos. Esto es lo contrario del resultado de 30 características de la edición Python, donde la PC1 en bruto llegaba al 98 % y era casi enteramente <em>worst area</em> en µm². Aquí la PC1 en bruto está apenas dominada por <em>Bare.nuclei</em>, la característica con la mayor varianza en bruto, y la estandarización elimina incluso ese leve desequilibrio. La única gran primera componente —dos tercios de la varianza— refleja un hecho biológico real: las nueve puntuaciones citológicas están fuertemente correlacionadas de forma positiva, porque un núcleo maligno tiende a ser anormal en todas ellas a la vez.</p>

<div class="box key"><h4>Estandarizar antes de PCA</h4><p>PCA maximiza la varianza, y la varianza tiene unidades. Una característica medida en µm² tiene una varianza en µm⁴ y dominará a las características medidas como cocientes, cualquiera sea su importancia biológica. A menos que todas las características compartan una unidad y sus varianzas sean comparables de manera significativa —como, excepcionalmente, ocurre aquí—, estandariza primero, lo que equivale a realizar PCA sobre la matriz de <em>correlación</em> (<code>prcomp(..., scale. = TRUE)</code>) en lugar de la matriz de covarianza. La revisión de Jolliffe y Cadima discute esta y las demás decisiones prácticas del método.<sup class="c"><a href="#r21">[21]</a></sup></p></div>

### 4.4 Cargas y el biplot

Los elementos de cada vector propio son las <strong>cargas</strong> (loadings): el peso de cada característica original en la componente. Leerlas es la forma de interpretar una componente. El <strong>biplot</strong> superpone las puntuaciones de los casos en las dos primeras componentes (puntos) con las cargas de las características (flechas): una flecha larga significa que la característica contribuye fuertemente; flechas que apuntan en la misma dirección indican características correlacionadas; la posición de un caso a lo largo de una flecha indica un valor alto o bajo de esa característica.



In [ ]:
load <- pca_std$rotation[, 1:2]
cat("|loading| on PC1 (sorted):\n"); print(round(sort(abs(load[, 1]), decreasing = TRUE), 3))
cat("largest |loading| on PC2:\n"); print(round(sort(abs(load[, 2]), decreasing = TRUE)[1:3], 3))
sc <- pca_std$x[, 1:2]
par(mar = c(4, 4, 2, 1))
plot(sc, col = GREY, pch = 19, cex = 0.4, xlab = sprintf("PC1 (%.0f%%)", 100 * vr_std[1]),
     ylab = sprintf("PC2 (%.0f%%)", 100 * vr_std[2]), main = "Biplot of the standardised breast-cancer data")
abline(h = 0, v = 0, col = GREY, lwd = 0.5); scl <- 4.5
arrows(0, 0, scl * load[, 1], scl * load[, 2], col = ORANGE, lwd = 1.4, length = 0.08)
text(scl * load[, 1] * 1.14, scl * load[, 2] * 1.14, seq_len(nrow(load)), col = ORANGE, font = 2, cex = 0.9)  # puntas numeradas
legend("topright", legend = paste0(seq_len(nrow(load)), ". ", rownames(load)), text.col = ORANGE, bty = "n", cex = 0.72, title = "feature loadings")



<p class="caption"><b>Figura 11.</b> Biplot. Los 683 casos (gris) se grafican según sus puntuaciones en PC1 y PC2; las flechas naranjas son las cargas de las nueve características (numeradas, con la clave a la derecha), escaladas para que sean visibles. Ocho de las nueve flechas apuntan casi en la misma dirección a lo largo de PC1: <em>cell size</em>, <em>cell shape</em>, <em>bland chromatin</em>, <em>epithelial cell size</em>, <em>normal nucleoli</em>, <em>bare nuclei</em>, <em>marginal adhesion</em> y <em>clump thickness</em> cargan todas en PC1 con pesos similares (0.30–0.38), de modo que PC1 es esencialmente el <em>promedio</em> de las puntuaciones citológicas: un eje general de "anormalidad nuclear", y la dirección a lo largo de la cual los casos benignos (izquierda) se separan de los malignos (derecha). La novena característica, <em>Mitoses</em>, apunta casi en línea recta hacia arriba: domina PC2 (carga 0.91) porque es la puntuación menos correlacionada con las demás, alta solo en una minoría de tumores agresivos. No se usó ninguna etiqueta y, sin embargo, el diagnóstico oculto es en gran medida una función de PC1, que es la razón por la que k-means lo encontró en la Sección 2.8.</p>

La tabla de cargas confirma esta lectura: las ocho puntuaciones de tamaño y textura cargan en PC1 con un peso casi igual, de modo que PC1 se aproxima a su media, mientras que PC2 es casi enteramente <em>Mitoses</em>. Una interpretación de este tipo es el principal uso científico de PCA: le dice al analista que las nueve características miden, en una primera aproximación, dos cosas: un grado dominante de anormalidad y una señal mitótica en gran medida independiente.

### 4.5 PCA como descomposición en valores singulares; blanqueo

La función <code>prcomp</code> de R no forma la matriz de covarianza. Calcula la <strong>descomposición en valores singulares</strong> (SVD) de la matriz de datos centrada, \(\mathbf{X} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^\top\), donde \(\mathbf{V}\) contiene los vectores singulares derechos (las componentes principales, como columnas), \(\boldsymbol{\Sigma}\) los valores singulares \(\sigma_j\) y \(\mathbf{U}\boldsymbol{\Sigma}\) las puntuaciones. La conexión es \(\mathbf{X}^\top \mathbf{X} = \mathbf{V} \boldsymbol{\Sigma}^2 \mathbf{V}^\top\), de modo que los valores propios de \(\mathbf{S}\) son \(\lambda_j = \sigma_j^2 / (n - 1)\). La vía de la SVD es numéricamente más estable y evita la matriz \(p \times p\), lo que importa cuando \(p\) está en los miles (genes).

El <strong>blanqueo</strong> (whitening, o esferización) va un paso más allá: divide cada puntuación por su desviación estándar \(\sqrt{\lambda_j}\), de modo que los datos transformados tengan covarianza identidad: toda dirección tiene varianza unitaria y todas son no correlacionadas. El blanqueo descarta la información de varianza según la cual PCA ordena las componentes, lo que lo hace inútil para elegir componentes pero esencial como primer paso de ICA (Sección 5).



In [ ]:
Xc <- scale(X_std, center = TRUE, scale = FALSE); sv <- svd(Xc)
cat("eigenvalues from SVD, sigma^2/(n-1):", round(sv$d[1:4]^2 / (nrow(Xc) - 1), 3), "\n")
cat("eigenvalues from prcomp:            ", round(prcomp(X_std)$sdev[1:4]^2, 3), "\n")
pr <- prcomp(X_std); scr <- pr$x[, 1:5]; whi <- sweep(scr, 2, pr$sdev[1:5], "/")
cat("covariance of the first five scores (diagonal = eigenvalues, off-diagonal ~ 0):\n"); print(round(cov(scr), 2))
cat("covariance of the five whitened scores (identity):\n"); print(round(cov(whi), 2))



Los valores singulares al cuadrado y divididos por \(n - 1\) coinciden con los valores propios en todos los dígitos impresos. La covarianza de las puntuaciones es diagonal —las componentes son no correlacionadas por construcción, con los valores propios en la diagonal— y, tras el blanqueo, es la identidad.

### 4.6 PCA para visualización y como preprocesamiento

De aquí se siguen dos usos prácticos. Primero, la <strong>visualización</strong>: proyectar datos de alta dimensión sobre las dos primeras componentes da la imagen bidimensional que retiene la mayor varianza. R no tiene un conjunto de datos de dígitos manuscritos (la edición Python usó <code>load_digits</code> de scikit-learn<sup class="c"><a href="#r22">[22]</a></sup>), de modo que el bloque de código <em>simula</em> seis clases de imágenes de 8 × 8 —una barra vertical, una barra horizontal, una diagonal, un bloque relleno, un marco y una antidiagonal, cada una con ruido en los píxeles— y las proyecta, exactamente como la edición Python proyectó los dígitos reales.



In [ ]:
set.seed(0)
templ <- list(function(){m<-matrix(0,8,8); m[,4:5]<-1; c(m)}, function(){m<-matrix(0,8,8); m[4:5,]<-1; c(m)},
              function(){m<-matrix(0,8,8); diag(m)<-1; c(m)}, function(){m<-matrix(0,8,8); m[3:6,3:6]<-1; c(m)},
              function(){m<-matrix(0,8,8); m[1,]<-m[8,]<-m[,1]<-m[,8]<-1; c(m)}, function(){m<-matrix(0,8,8); m[cbind(1:8,8:1)]<-1; c(m)})
dig <- do.call(rbind, lapply(templ, function(f) t(replicate(60, f() + rnorm(64, 0, 0.15))))); dlab <- rep(1:6, each = 60)
pd <- prcomp(dig); Zd <- pd$x[, 1:2]
cat("variance retained by two components:", round(sum((pd$sdev^2 / sum(pd$sdev^2))[1:2]), 3),
    "| components for 90%:", which(cumsum(pd$sdev^2 / sum(pd$sdev^2)) >= 0.9)[1], "of 64\n")
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(Zd, col = PALETTE[dlab], pch = 19, cex = 0.5, xlab = "PC1", ylab = "PC2", main = "simulated images in the PC1-PC2 plane")
for (d in 1:6) text(mean(Zd[dlab==d, 1]), mean(Zd[dlab==d, 2]), d, font = 2, cex = 1.3)
image(1:8, 1:8, matrix(pd$rotation[, 1], 8, 8)[, 8:1], col = colorRampPalette(c(BLUE, "white", ORANGE))(30),
      xlab = "", ylab = "", main = "PC1 as an 8x8 image", axes = FALSE)



<p class="caption"><b>Figura 12.</b> Izquierda: las 360 imágenes simuladas en el plano de sus dos primeras componentes, coloreadas y numeradas según la clase de patrón (oculta para PCA). Dos componentes retienen apenas algo más de la mitad de la varianza y, sin embargo, las seis clases ya forman regiones separadas. Derecha: la primera componente mostrada como una imagen de 8 × 8; una componente de un conjunto de datos de imágenes es en sí misma una imagen, y leerla así es el equivalente, en el espacio de píxeles, de leer una tabla de cargas. Los patrones son simulados; la pedagogía (PCA para visualización, componentes como imágenes) es idéntica a la de los dígitos manuscritos de la edición Python.</p>

Segundo, el <strong>preprocesamiento</strong>: reemplazar las \(p\) características por las primeras \(q\) componentes antes de un clasificador supervisado elimina redundancia, reduce el ruido y acelera los métodos basados en distancias como k-NN. El bloque de código mide la exactitud con validación cruzada de 5 particiones (k-fold) de un 5-NN en los datos de cáncer de mama en función del número de componentes retenidas, con el PCA ajustado <em>dentro</em> de cada partición, porque ajustarlo primero con todos los datos permitiría que las particiones de prueba influyeran en las componentes (la fuga de datos del Suplemento S1, en una forma que a menudo se pasa por alto porque PCA no usa etiquetas).



In [ ]:
suppressPackageStartupMessages(library(class))
set.seed(0); folds <- sample(rep(1:5, length.out = nrow(X_raw)))
cv_knn_pca <- function(q) mean(sapply(1:5, function(f) {
  tr <- folds != f; ctr <- colMeans(X_raw[tr, ]); sdt <- apply(X_raw[tr, ], 2, sd)
  Xtr <- scale(X_raw[tr, ], ctr, sdt); Xte <- scale(X_raw[!tr, ], ctr, sdt)
  if (q < 9) { p <- prcomp(Xtr); Xtr <- Xtr %*% p$rotation[, 1:q, drop = FALSE]; Xte <- Xte %*% p$rotation[, 1:q, drop = FALSE] }
  mean(knn(Xtr, Xte, y_hidden[tr], k = 5) == y_hidden[!tr])
}))
qs <- c(1, 2, 3, 4, 5, 7, 9); accs <- sapply(qs, cv_knn_pca)
print(data.frame(components = qs, CV_accuracy = round(accs, 4)))
par(mar = c(4, 4, 2, 1)); plot(qs, accs, type = "b", pch = 19, col = BLUE, ylim = c(0.95, 0.98),
     xlab = "number of principal components q", ylab = "5-fold CV accuracy", main = "k-NN accuracy after reducing to q components")
abline(h = accs[length(accs)], col = ORANGE, lty = 2)



<p class="caption"><b>Figura 13.</b> Exactitud con validación cruzada de un 5-NN entrenado con las primeras \(q\) componentes principales de las características estandarizadas. En estos datos, una <em>sola</em> componente —el eje de anormalidad general de la Figura 11— ya da alrededor del 97 %, igualando o superando levemente a las nueve características completas (línea discontinua), y la curva es esencialmente plana a partir de ahí. Ocho de las nueve dimensiones pueden descartarse sin costo, porque la señal diagnóstica reside casi por completo en PC1; las nueve puntuaciones citológicas son, para esta tarea, efectivamente unidimensionales. (Los datos de 30 características de la edición Python necesitaban unas cinco componentes para alcanzar la meseta: un recordatorio de que el número de dimensiones útiles es una propiedad del conjunto de datos, no algo universal.)</p>

Observa el uso de las etiquetas aquí. PCA en sí no usó ninguna —las componentes son las mismas exista o no \(\mathbf{y}\)—, pero la <em>evaluación</em> de cuántas componentes conservar para una tarea supervisada necesariamente usa las etiquetas, a través de la exactitud con validación cruzada. Eso es legítimo siempre que el PCA se vuelva a ajustar dentro de cada partición, como hace el bloque de código.

### 4.7 Límites de PCA, y t-SNE y UMAP para visualización

PCA es <strong>lineal</strong>: las componentes son sumas ponderadas de las características y el subespacio es plano. Los datos que yacen sobre una superficie curva —una espiral, una esfera, las medias lunas— quedan mal resumidos por cualquier subespacio plano. El PCA con kernel (kernel PCA) y los autoencoders de la Unidad 6 son las extensiones no lineales.

Más importante, y más a menudo olvidado: <strong>la varianza no es información</strong>. PCA ordena las direcciones según cuánto se dispersan los datos a lo largo de ellas, no según cuán relevante es esa dispersión para alguna pregunta. Si la diferencia entre dos grupos se encuentra a lo largo de una dirección de poca varianza mientras domina una fuente de variación grande e irrelevante (edad, lote, instrumento), las primeras componentes describirán la fuente irrelevante y la señal quedará en una componente tardía o será descartada. El bloque de código construye exactamente esta situación.



In [ ]:
set.seed(0); n <- 300
irrelevant <- rnorm(n, 0, 4)                                  # p. ej., edad del paciente: gran dispersión, no relacionada con los grupos
group <- sample(0:1, n, replace = TRUE)
marker <- ifelse(group == 0, -1, 1) + rnorm(n, 0, 0.4)        # característica informativa: poca dispersión, separa los grupos
Xvi <- cbind(irrelevant, marker); pv <- prcomp(Xvi); Zvi <- pv$x
cat("proportion of variance:", round(pv$sdev^2 / sum(pv$sdev^2), 3), "| PC1 loadings:", round(pv$rotation[, 1], 3), "\n")
cv_glm <- function(F) { set.seed(0); fo <- sample(rep(1:5, length.out = n)); mean(sapply(1:5, function(f) {
  tr <- fo != f; m <- glm(group[tr] ~ ., data.frame(F[tr, , drop = FALSE]), family = binomial)
  mean((predict(m, data.frame(F[!tr, , drop = FALSE]), type = "response") > 0.5) == group[!tr]) })) }
cat(sprintf("CV accuracy of logistic regression:  PC1 only = %.3f | PC2 only = %.3f | both features = %.3f\n",
            cv_glm(Zvi[, 1, drop = FALSE]), cv_glm(Zvi[, 2, drop = FALSE]), cv_glm(Xvi)))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1)); gc <- ifelse(group == 0, BLUE, ORANGE)
plot(Xvi, col = gc, pch = 19, cex = 0.5, xlab = "irrelevant feature (variance 16)", ylab = "marker (variance ~1.2)", main = "original features")
plot(Zvi, col = gc, pch = 19, cex = 0.5, xlab = "PC1 (93%)", ylab = "PC2 (7%)", main = "principal components")



<p class="caption"><b>Figura 14.</b> Izquierda: dos grupos (colores ocultos para PCA) separados nítidamente a lo largo de una característica de poca varianza, mientras que una característica no relacionada tiene alrededor de catorce veces esa varianza. Derecha: PCA pone alrededor del 93 % de la varianza en PC1, que es esencialmente la característica irrelevante; la separación entre grupos sobrevive solo en PC2. Un analista que conservara "la componente que explica el 93 % de la varianza" habría descartado la única dirección informativa: una regresión logística con solo PC1 rinde cerca del nivel del azar (alrededor del 57 %), mientras que una con solo PC2 es casi perfecta, como muestra la salida impresa.</p>

Para la <strong>visualización</strong> de datos de alta dimensión en dos dimensiones, dos métodos no lineales han desplazado en gran medida a PCA en la literatura biomédica: <strong>t-SNE</strong>,<sup class="c"><a href="#r23">[23]</a></sup> que sitúa los puntos de modo que los vecinos en el espacio original sigan siendo vecinos, y <strong>UMAP</strong>,<sup class="c"><a href="#r24">[24]</a></sup> que hace lo mismo más rápido y con una pretensión más fuerte de preservar la estructura gruesa. Ambos producen los llamativos gráficos de "islas" de la secuenciación de ARN de célula única (paquetes de R <code>Rtsne</code> y <code>uwot</code>). Ambos son <em>solo para visualización</em>: las distancias entre islas y sus tamaños no tienen significado, el resultado depende de parámetros de ajuste y de la semilla aleatoria, y no se pueden ubicar puntos nuevos sin recalcular. Nunca deben usarse como preprocesamiento para agrupamiento o clasificación, y los clusters que se vean en ellos deben confirmarse en el espacio de características original.

<div class="box warn"><h4>Error frecuente</h4><p>"Las dos primeras componentes explican solo el 30 % de la varianza, así que el PCA falló." Una proporción baja en dos componentes significa que los datos son genuinamente de alta dimensión, lo cual es un hallazgo, no una falla; y, como muestra la Figura 14, una proporción alta en una componente puede ser una falla, cuando esa componente es irrelevante. La proporción de varianza mide compresión, no relevancia.</p></div>

## 5. Análisis de componentes independientes

### 5.1 El problema del cóctel; independencia frente a no correlación

Dos personas hablan a la vez en una sala con dos micrófonos. Cada micrófono registra una suma ponderada distinta de las dos voces, con pesos que dependen de las distancias. Dadas solo las dos grabaciones, ¿pueden recuperarse las dos voces? Este es el <strong>problema del cóctel</strong> (cocktail-party problem), el prototipo de la <strong>separación ciega de fuentes</strong>: \(m\) señales observadas \(\mathbf{x}(t)\) se modelan como una mezcla lineal desconocida de \(m\) señales fuente desconocidas \(\mathbf{s}(t)\),

<div class="eq">\[ \mathbf{x} = \mathbf{A}\,\mathbf{s}, \qquad \hat{\mathbf{s}} = \mathbf{W}\,\mathbf{x} \]
<span class="where">donde \(\mathbf{A}\) es la <em>matriz de mezcla</em> \(m \times m\), y la tarea consiste en encontrar una <em>matriz de separación</em> \(\mathbf{W} \approx \mathbf{A}^{-1}\) a partir solo de las observaciones, sin conocer \(\mathbf{A}\) ni las fuentes salvo por un supuesto: las fuentes son <strong>estadísticamente independientes</strong>.</span></div>

El análisis de componentes independientes (ICA) es la familia de métodos que resuelven este problema. El nombre y la definición formal se deben a Comon (1994);<sup class="c"><a href="#r25">[25]</a></sup> el algoritmo que lo hizo práctico, Infomax, a Bell y Sejnowski (1995);<sup class="c"><a href="#r26">[26]</a></sup> y el algoritmo rápido de punto fijo que se implementa más abajo, FastICA, a Hyvärinen y Oja, cuya revisión de 2000 es la introducción estándar.<sup class="c"><a href="#r27">[27]</a></sup>

La distinción que hace a ICA diferente de PCA es la que existe entre <em>no correlacionado</em> e <em>independiente</em>. Dos variables son no correlacionadas cuando su covarianza es cero, una afirmación solo sobre los momentos de segundo orden. Son independientes cuando su distribución conjunta se factoriza, \(p(s_1, s_2) = p(s_1)\,p(s_2)\), una afirmación sobre todos los momentos. La independencia implica la no correlación; la recíproca se cumple solo para variables gaussianas. PCA encuentra componentes no correlacionadas, y para una señal gaussiana eso es todo lo que hay que encontrar; para señales no gaussianas existe una estructura más fuerte, la independencia, y PCA no la ve.

### 5.2 La no gaussianidad como palanca

¿Por qué puede recuperarse la independencia? El argumento es el teorema central del límite al revés. Una suma de variables aleatorias independientes es <em>más gaussiana</em> que cualquiera de sus términos. Cada mezcla observada \(x_j = \sum_l a_{jl} s_l\) es una suma de ese tipo, de modo que las mezclas están más cerca de lo gaussiano que las fuentes. Recíprocamente, entre todas las combinaciones lineales \(\mathbf{w}^\top \mathbf{x}\) de las observaciones, las <em>menos</em> gaussianas deben ser las fuentes originales (salvo por la escala). ICA busca, por lo tanto, las direcciones \(\mathbf{w}\) que maximizan una medida de no gaussianidad, y funciona exactamente en la medida en que las fuentes son no gaussianas: <strong>a lo sumo una fuente puede ser gaussiana</strong>, o el problema no tiene solución única.

Se usan dos medidas de no gaussianidad. La <strong>curtosis</strong>, el cuarto momento estandarizado \(\kappa = \mathbb{E}[s^4]/\mathbb{E}[s^2]^2 - 3\), es cero para una gaussiana, positiva para distribuciones "puntiagudas" de colas pesadas (voz, espigas) y negativa para las planas (una onda cuadrada, una variable uniforme); es simple pero sensible a valores atípicos. La <strong>negentropía</strong>, el déficit de entropía respecto de una gaussiana de igual varianza, es cero para una gaussiana y positiva en otro caso, es robusta, y es lo que FastICA aproxima con una función de contraste suave como \(\log\cosh\) (cuya derivada es \(\tanh\)). FastICA primero <em>blanquea</em> los datos con PCA (Sección 4.5), de modo que el problema restante es encontrar una <em>rotación</em>, y luego encuentra la rotación una dirección a la vez mediante una iteración de punto fijo sobre el contraste elegido.

### 5.3 FastICA desde cero con tres fuentes sintéticas: PCA falla, ICA tiene éxito

R base no tiene FastICA, de modo que el bloque de código lo implementa desde cero en unas veinte líneas —centrar, blanquear mediante <code>eigen</code> de la covarianza y luego una iteración de punto fijo con \(g = \tanh\) y deflación de Gram–Schmidt—, tal como la edición Python escribió el suyo. Tres fuentes —una sinusoide, una onda cuadrada y un diente de sierra— se mezclan mediante una matriz aleatoria de \(3 \times 3\) en tres canales observados. Se pide a PCA y a FastICA tres componentes a cada uno, y las componentes recuperadas se comparan con las fuentes verdaderas mediante su correlación absoluta.



In [ ]:
fastica <- function(X, n_comp, max_iter = 200, tol = 1e-6, seed = 1) {
  Xc <- scale(X, center = TRUE, scale = FALSE)               # 1. centrar
  e  <- eigen(cov(Xc)); K <- diag(1 / sqrt(e$values)) %*% t(e$vectors)   # 2. matriz de blanqueo (PCA)
  Xw <- (Xc %*% t(K))[, 1:n_comp, drop = FALSE]              #    datos blanqueados: covarianza identidad
  W  <- matrix(0, n_comp, n_comp); set.seed(seed)
  for (i in 1:n_comp) {                                       # 3. una dirección independiente a la vez
    w <- rnorm(n_comp); w <- w / sqrt(sum(w^2))
    for (it in 1:max_iter) {
      wx <- as.vector(Xw %*% w); g <- tanh(wx); gp <- 1 - g^2
      wn <- colMeans(g * Xw) - mean(gp) * w                   #    actualización de punto fijo (negentropía, g = tanh)
      if (i > 1) { Wp <- W[1:(i - 1), , drop = FALSE]; wn <- wn - t(Wp) %*% (Wp %*% wn) }  # deflación
      wn <- as.vector(wn); wn <- wn / sqrt(sum(wn^2))
      if (abs(abs(sum(wn * w)) - 1) < tol) { w <- wn; break }
      w <- wn
    }
    W[i, ] <- w
  }
  list(S = Xw %*% t(W), U = W %*% K[1:n_comp, , drop = FALSE], mu = colMeans(X))   # S = fuentes; U = separación completa
}
set.seed(0); tt <- seq(0, 8, length.out = 2000)
saw <- function(x) 2 * (x / (2 * pi) - floor(0.5 + x / (2 * pi)))
S_true <- cbind(sin(2 * pi * tt), sign(sin(2 * pi * 0.6 * tt + 1)), saw(2 * pi * 0.4 * tt))
S_true <- scale(S_true + 0.05 * matrix(rnorm(length(S_true)), ncol = 3))
A_mix  <- matrix(rnorm(9), 3, 3); X_mix <- S_true %*% t(A_mix)
kurt <- function(M) apply(M, 2, function(z) mean(((z - mean(z)) / sd(z))^4) - 3)
cat("kurtosis of the sources :", round(kurt(S_true), 2), " (far from 0: non-Gaussian)\n")
cat("kurtosis of the mixtures:", round(kurt(X_mix), 2), " (closer to 0: mixing Gaussianises)\n")
S_pca <- prcomp(X_mix)$x; S_ica <- fastica(X_mix, 3)$S
mt <- function(R, nm) { C <- abs(cor(S_true, R)); dimnames(C) <- list(c("sine","square","saw"), paste0(nm, 1:3)); round(C, 3) }
cat("\nPCA |correlation| with each source:\n"); print(mt(S_pca, "PC"))
cat("\nICA |correlation| with each source:\n"); print(mt(S_ica, "IC"))



La salida de la curtosis muestra en acción el argumento del límite central: cada fuente tiene una curtosis muy por debajo de cero (una sinusoide, una onda cuadrada y un diente de sierra son todas de techo plano), y cada mezcla está más cerca de cero. Las tablas de correlación muestran el resultado. Cada componente de PCA es una combinación de las tres fuentes, con correlaciones repartidas a lo largo de la fila (la mejor por fuente, solo 0.75–0.89): PCA ha encontrado direcciones no correlacionadas, lo que no basta. Cada componente de ICA se correlaciona casi perfectamente (por encima de 0.98) con exactamente una fuente y de forma despreciable con las otras dos: las fuentes se han recuperado sin ningún conocimiento de la matriz de mezcla.



In [ ]:
panels <- list(list("true sources s(t)", S_true), list("observed mixtures x(t)", X_mix),
               list("PCA components", S_pca), list("ICA components", S_ica))
par(mfcol = c(3, 4), mar = c(2, 2, 2, 1))
for (col in seq_along(panels)) for (row in 1:3) {
  plot(tt, panels[[col]][[2]][, row], type = "l", col = PALETTE[col], lwd = 0.7, xaxt = "n", yaxt = "n", xlab = "", ylab = "")
  if (row == 1) title(panels[[col]][[1]], cex.main = 0.95)
}



<p class="caption"><b>Figura 15.</b> Columnas: las tres fuentes; las tres mezclas registradas, cada una una suma ponderada de todas las fuentes; las tres componentes principales, que siguen siendo mezclas (una sinusoide con una onda cuadrada superpuesta, etc.); y las tres componentes independientes, que vuelven a ser las fuentes, pero en otro orden y, algunas, invertidas. Compara el orden de las filas de la primera y la última columna.</p>

**Las dos ambigüedades.** ICA no puede recuperar el <em>orden</em> de las fuentes —cualquier permutación de las filas de \(\mathbf{s}\) con la permutación correspondiente de las columnas de \(\mathbf{A}\) da el mismo \(\mathbf{x}\)— ni su <em>escala y signo</em>, ya que multiplicar una fuente por \(c\) y su columna de \(\mathbf{A}\) por \(1/c\) no cambia nada. FastICA fija la escala devolviendo componentes de varianza unitaria y deja arbitrarios el orden y el signo. En consecuencia, la "primera componente independiente" no significa nada (a diferencia de la primera componente principal); las componentes se identifican por su contenido —forma de onda, patrón espacial, correlación con una referencia— y esa identificación es el punto en que entra el conocimiento del analista.

### 5.4 Aplicación biomédica: eliminar un artefacto de un registro multicanal

Los registros electrofisiológicos —el EEG con sus 19 a 256 electrodos de cuero cabelludo, el ECG de múltiples derivaciones, el magnetoencefalograma— son el terreno natural de ICA, porque cada electrodo registra una mezcla lineal, a través de los tejidos conductores, de varias fuentes: la actividad neural o cardíaca de interés y artefactos como parpadeos, actividad muscular, el latido cardíaco captado por el EEG, la deriva de los electrodos y la interferencia de la red eléctrica. Makeig y colaboradores aplicaron por primera vez ICA al EEG en 1996,<sup class="c"><a href="#r28">[28]</a></sup> y Jung y colaboradores mostraron en 2000 que las componentes de artefacto podían identificarse y eliminarse —el registro se reconstruye a partir de las componentes restantes— de forma más selectiva que rechazando las épocas contaminadas o mediante regresión sobre un canal de referencia.<sup class="c"><a href="#r29">[29]</a></sup> Este <em>rechazo de artefactos basado en ICA</em> es hoy un paso estándar en el preprocesamiento de EEG.

El bloque de código simula la situación en miniatura: una fuente "ECG" limpia, un artefacto parecido a un parpadeo (tres protuberancias gaussianas lentas) y una deriva lenta de la línea base se mezclan en cuatro canales con distintos pesos; se ejecuta FastICA con tres componentes; la componente del artefacto se identifica por su correlación con el parpadeo conocido (en la práctica: por su forma de onda y su patrón espacial), se anula y se reconstruyen los canales. La reconstrucción necesita la pseudoinversa de Moore–Penrose de la matriz de separación, <code>MASS::ginv</code>.<sup class="c"><a href="#r37">[37]</a></sup> La eliminación se evalúa con la raíz de la desviación cuadrática media de cada canal limpiado respecto de su versión sin parpadeo.



In [ ]:
set.seed(0); fs <- 250; tt2 <- seq(0, 10, by = 1 / fs); beats <- seq(0.4, 10, by = 0.8)   # 75 lpm
ecg   <- rowSums(sapply(beats, function(b) exp(-((tt2-b)/.012)^2) - .3*exp(-((tt2-b-.05)/.02)^2) + .25*exp(-((tt2-b+.2)/.06)^2)))
blink <- rowSums(sapply(c(1.5, 4.2, 7.8), function(b) exp(-((tt2-b)/.12)^2)))              # tres deflexiones lentas
drift <- 0.3 * sin(2 * pi * 0.2 * tt2)                                                     # oscilación de tipo respiratorio
src <- cbind(ecg, blink, drift)
Mmix <- matrix(c(1,.2,.3, .8,1.2,.2, .6,.6,.5, .3,1.5,.4), 4, 3, byrow = TRUE)             # 4 canales x 3 fuentes
channels <- src %*% t(Mmix) + 0.03 * matrix(rnorm(length(tt2) * 4), ncol = 4)
clean_target <- channels - outer(blink, Mmix[, 2])                                         # canales sin el parpadeo
ic <- fastica(channels, 3, max_iter = 1000, seed = 2)
corr <- abs(cor(src, ic$S)); dimnames(corr) <- list(c("ecg","blink","drift"), paste0("IC", 1:3))
k_art <- which.max(corr["blink", ])                                                        # identificar la componente del parpadeo
S_clean <- ic$S; S_clean[, k_art] <- 0
cleaned <- sweep(S_clean %*% t(MASS::ginv(ic$U)), 2, ic$mu, "+")                            # anularla y reconstruir
rmse <- function(a, b) sqrt(colMeans((a - b)^2))
cat("|corr| of components with (ecg, blink, drift):\n"); print(round(corr, 3))
cat(sprintf("\nartefact component identified: IC%d\n", k_art))
print(data.frame(blink_weight = Mmix[, 2], RMSE_before = round(rmse(channels, clean_target), 3),
                 RMSE_after = round(rmse(cleaned, clean_target), 3), row.names = paste0("ch", 1:4)))


In [ ]:
layout(matrix(c(1,5, 2,6, 3,7, 4,8), 4, 2, byrow = TRUE), widths = c(2.2, 1))
par(mar = c(2, 3, 1.5, 1))
for (j in 1:4) {
  plot(tt2, channels[, j], type = "l", col = GREY, lwd = 0.6, xlab = "", ylab = paste0("ch", j), yaxt = "n")
  lines(tt2, cleaned[, j], col = BLUE, lwd = 0.6)
  for (b in c(1.5, 4.2, 7.8)) rect(b - .3, par("usr")[3], b + .3, par("usr")[4], col = rgb(.82,.41,.12,.12), border = NA)
  if (j == 1) legend("topright", c("recorded", "after removing artefact IC"), col = c(GREY, BLUE), lwd = 1.5, bty = "n", cex = 0.8)
}
nm <- c("ecg", "blink", "drift")
for (j in 1:3) { plot(tt2, ic$S[, j], type = "l", lwd = 0.6, col = if (j == k_art) ORANGE else GREEN, xlab = "", ylab = paste0("IC", j), yaxt = "n")
                 title(sprintf("IC%d ~ %s", j, nm[which.max(corr[, j])]), cex.main = 0.85) }
plot.new()



<p class="caption"><b>Figura 16.</b> Izquierda: los cuatro canales registrados (gris) y los mismos canales después de anular la componente del artefacto y reconstruir el registro (azul). Los tres intervalos sombreados marcan los parpadeos; en los canales registrados aparecen como deflexiones lentas de tamaño variable —grandes en los canales 2 y 4, que tienen el mayor peso de parpadeo— y en los canales limpiados han desaparecido en gran medida, mientras que los latidos quedan intactos. Derecha: las tres componentes independientes, una de las cuales (naranja) es el parpadeo; las otras son el ECG y la deriva. La tabla informa la raíz de la desviación cuadrática media de cada canal respecto de su versión sin parpadeo, que disminuye en un factor de dos a tres; el residuo es el precio de una separación imperfecta en presencia de ruido y de la indeterminación de signo/escala absorbida en la reconstrucción.</p>

<div class="box key"><h4>¿Por qué ICA y no filtrado?</h4><p>Un parpadeo y un latido se superponen en frecuencia, de modo que ningún filtro de frecuencia elimina uno sin distorsionar el otro. ICA los separa por una propiedad distinta —su independencia estadística y sus diferentes firmas <em>espaciales</em> sobre los canales—, y por eso necesita varios canales mientras que un filtro necesita uno. La misma lógica se aplica a la eliminación del artefacto cardíaco del EEG, del artefacto muscular y del ECG materno en un registro de ECG fetal abdominal.</p></div>

### 5.5 PCA frente a ICA

| | PCA | ICA |
|---|---|---|
| Criterio | Varianza máxima / mínimo error de reconstrucción | Máxima independencia estadística (no gaussianidad) |
| Estadísticos usados | Solo de segundo orden (covarianza) | De orden superior (curtosis, negentropía) |
| Las componentes son | No correlacionadas y <em>ortogonales</em> | Independientes; en general no ortogonales |
| Orden | Según la varianza; la primera componente es especial | Ninguno; las componentes se identifican por su contenido |
| Signo / escala | Signo arbitrario; escala \(= \sqrt{\lambda}\) | Signo y escala arbitrarios (varianza unitaria por convención) |
| Solución | Única, de forma cerrada (descomposición en valores propios / SVD) | Iterativa; depende del inicio y del contraste; a lo sumo una fuente gaussiana |
| Uso típico | Compresión, visualización, reducción de ruido, preprocesamiento | Separación ciega de fuentes: artefactos de EEG/ECG/MEG, redes en fMRI, separación espectral (unmixing) hiperespectral |
| Supone | Linealidad; varianza = relevancia | Linealidad; fuentes independientes y no gaussianas; tantos canales como fuentes |

## 6. Otros métodos, brevemente

**Modelos de mezclas gaussianas y el algoritmo EM.** Modelar los datos como una suma ponderada de \(k\) densidades gaussianas, \(p(\mathbf{x}) = \sum_j \pi_j\, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)\), y ajustar los pesos, medias y covarianzas por máxima verosimilitud. El algoritmo de ajuste es el de <strong>esperanza–maximización</strong> (EM),<sup class="c"><a href="#r30">[30]</a></sup> que alterna un paso E —calcular, para cada punto, la probabilidad posterior (<em>responsabilidad</em>) de cada componente— y un paso M —reestimar cada componente como estadísticos ponderados por la responsabilidad—. Esto es exactamente un <em>k-means blando</em>: k-means es el límite de EM para gaussianas con covarianzas esféricas iguales cuando las responsabilidades se fuerzan a 0 o 1. Como cada componente tiene su propia covarianza, las mezclas manejan clusters de distinto tamaño, dispersión y orientación (la falla de la Figura 5a), y el número de componentes puede elegirse mediante un criterio de verosimilitud (BIC). En R lo proporciona el paquete <code>mclust</code>.

**DBSCAN.** El agrupamiento espacial basado en densidad de aplicaciones con ruido (density-based spatial clustering of applications with noise)<sup class="c"><a href="#r31">[31]</a></sup> define un cluster como un conjunto maximal de puntos conectados a través de regiones de alta densidad: un punto es un punto <em>núcleo</em> (core) si al menos <code>minPts</code> puntos se encuentran a distancia <code>eps</code> o menos de él; los puntos núcleo a distancia <code>eps</code> entre sí están en el mismo cluster; los puntos no núcleo al alcance de un punto núcleo se unen a él; todo lo demás se etiqueta como <em>ruido</em>. No necesita \(k\), encuentra clusters de cualquier forma y es el único método aquí que puede negarse a asignar un punto. Sus debilidades son la sensibilidad a <code>eps</code> y su única escala de densidad. Lo proporciona el paquete de R <code>dbscan</code>.

**Agrupamiento espectral.** Construir un grafo de similitud sobre los puntos (pesos de k vecinos más cercanos o de kernel gaussiano), calcular los primeros vectores propios del laplaciano del grafo y ejecutar k-means sobre las filas de la matriz de vectores propios. Los vectores propios incrustan los datos en un espacio donde las regiones conectadas del grafo se convierten en grupos compactos, de modo que k-means puede entonces encontrar clusters de forma arbitraria; el método es una relajación del problema de corte de grafos. Es preciso para \(n\) moderados y costoso más allá, debido a la descomposición en valores propios.

Ni <code>mclust</code> ni <code>dbscan</code> vienen preinstalados en este entorno, de modo que —como habría hecho la edición Python si scikit-learn no los tuviera— el bloque de código demuestra la idea de <em>conectividad</em> con los dos métodos que R base sí ofrece: el agrupamiento jerárquico de enlace simple (Sección 3.5) y un <strong>agrupamiento espectral</strong> implementado desde cero en una docena de líneas. k-means representa a los métodos basados en centroides. Los tres se ejecutan sobre las medias lunas.



In [ ]:
spectral <- function(X, k, nn = 10) {                     # agrupamiento espectral con laplaciano normalizado, desde cero
  D <- as.matrix(dist(X)); n <- nrow(X); sigma <- median(D)
  W <- exp(-D^2 / (2 * sigma^2))
  for (i in 1:n) { th <- sort(D[i, ])[nn + 1]; W[i, D[i, ] > th] <- 0 }   # conservar los nn vecinos más cercanos
  W <- pmax(W, t(W)); dg <- rowSums(W)
  L <- diag(n) - diag(1 / sqrt(dg)) %*% W %*% diag(1 / sqrt(dg))          # laplaciano normalizado
  U <- eigen(L, symmetric = TRUE)$vectors[, (n - k + 1):n]               # vectores de los k menores valores propios
  U <- U / sqrt(rowSums(U^2) + 1e-9); kmeans(U, k, nstart = 10)$cluster
}
set.seed(0); X_mo <- make_moons(300, 0.06); y_mo <- rep(1:2, c(150, 150))
methods <- list("k-means (centroid)" = kmeans(X_mo, 2, nstart = 10)$cluster,
                "single linkage (connectivity)" = cutree(hclust(dist(X_mo), "single"), 2),
                "spectral, 10-NN graph" = spectral(X_mo, 2))
par(mfrow = c(1, 3), mar = c(2, 2, 3, 1))
for (nm in names(methods)) { lab <- methods[[nm]]
  plot(X_mo, col = PALETTE[lab], pch = 19, cex = 0.5, asp = 1, xlab = "", ylab = "",
       main = sprintf("%s\nARI = %.2f", nm, adjusted_rand(y_mo, lab))) }



<p class="caption"><b>Figura 17.</b> Las dos medias lunas con tres métodos. k-means falla (ARI de alrededor de 0.2): solo puede separar con una recta, y corta cada medialuna por la mitad. El agrupamiento de enlace simple y el agrupamiento espectral implementado desde cero recuperan ambos las dos lunas (ARI cercano a 1), porque ambos definen los clusters mediante la <em>conectividad</em> —cadenas de puntos cercanos— en lugar de la distancia a un centro. Una mezcla gaussiana mejoraría a k-means pero igual fallaría (solo puede separar con una curva cuadrática); DBSCAN tendría éxito, por la misma razón de conectividad que los dos métodos mostrados. La lección es otra vez la de la Sección 2.7: el método codifica un supuesto sobre la forma de los clusters, y el supuesto debe ajustarse a los datos.</p>

## 7. Evaluar resultados no supervisados

La Sección 1.2 planteó el problema: no hay una verdad con la cual comparar. Tres tipos de evidencia la sustituyen.

Los **índices internos** miden la geometría de la partición a partir solo de los datos. La <strong>silueta</strong> (Sección 2.6) premia las particiones cuyos puntos están más cerca de su propio cluster que del otro más cercano. El <strong>índice de Davies–Bouldin</strong><sup class="c"><a href="#r32">[32]</a></sup> es el promedio, sobre los clusters, del peor cociente entre la dispersión intra-cluster y la distancia entre centroides, \(\frac{1}{k}\sum_j \max_{l \ne j} \frac{\sigma_j + \sigma_l}{\lVert \boldsymbol{\mu}_j - \boldsymbol{\mu}_l \rVert}\); menor es mejor y 0 es lo ideal. Todos estos índices prefieren clusters compactos, redondos y bien separados, lo que significa que prefieren los clusters que produce k-means y calificarán mal una partición correcta de las medias lunas basada en la conectividad. Un índice interno mide la concordancia con un ideal geométrico, no la corrección.

Los **índices externos** comparan la partición con un etiquetado de referencia cuando existe. El <strong>índice de Rand ajustado</strong> (Sección 2.8) cuenta los pares concordantes con corrección por azar;<sup class="c"><a href="#r13">[13]</a></sup> la <strong>información mutua normalizada</strong> (NMI) mide cuánto te dice conocer el cluster acerca de la etiqueta, normalizada a \([0, 1]\), y se prefiere cuando el número de clusters y el de clases difieren. Ambos son invariantes al reetiquetado.

**Estabilidad.** Una estructura real debería reaparecer cuando se perturba la muestra. Vuelve a ejecutar el agrupamiento sobre subconjuntos aleatorios y mide la concordancia (ARI) entre ejecuciones; una partición que cambia sustancialmente de una mitad de los datos a la otra describe la muestra, no la población. La estabilidad es necesaria, no suficiente: k-means sobre datos uniformes es bastante estable, y bastante carente de sentido. El bloque de código calcula los tres índices internos y externos para \(k = 2, \dots, 4\) sobre los datos de cáncer de mama estandarizados (la silueta mediante <code>cluster::silhouette</code><sup class="c"><a href="#r38">[38]</a></sup>) y luego mide la estabilidad por submuestreo.



In [ ]:
davies_bouldin <- function(X, lab) { k <- max(lab); cen <- t(sapply(1:k, function(j) colMeans(X[lab == j, , drop = FALSE])))
  s <- sapply(1:k, function(j) mean(sqrt(rowSums(sweep(X[lab == j, , drop = FALSE], 2, cen[j, ])^2))))
  mean(sapply(1:k, function(j) max(sapply(setdiff(1:k, j), function(l) (s[j] + s[l]) / sqrt(sum((cen[j, ] - cen[l, ])^2)))))) }
nmi <- function(a, b) { t <- table(a, b) / length(a); pa <- rowSums(t); pb <- colSums(t)
  H <- function(p) -sum(p[p > 0] * log(p[p > 0])); I <- sum(ifelse(t > 0, t * log(t / outer(pa, pb)), 0)); I / sqrt(H(pa) * H(pb)) }
d_std <- dist(X_std)
idx <- t(sapply(2:6, function(k) { set.seed(0); lab <- kmeans(X_std, k, nstart = 10)$cluster
  c(silhouette = mean(silhouette(lab, d_std)[, 3]), Davies_Bouldin = davies_bouldin(X_std, lab),
    ARI = adjusted_rand(y_hidden, lab), NMI = nmi(y_hidden, lab)) }))
rownames(idx) <- paste0("k=", 2:6); print(round(idx, 3))
set.seed(0)
for (k in c(2, 3, 4)) {
  aris <- sapply(1:20, function(r) {
    i1 <- sample(nrow(X_std), nrow(X_std) / 2); i2 <- sample(nrow(X_std), nrow(X_std) / 2); common <- intersect(i1, i2)
    k1 <- kmeans(X_std[i1, ], k, nstart = 5); k2 <- kmeans(X_std[i2, ], k, nstart = 5)
    p1 <- max.col(-sapply(1:k, function(j) rowSums(sweep(X_std[common, ], 2, k1$centers[j, ])^2)))
    p2 <- max.col(-sapply(1:k, function(j) rowSums(sweep(X_std[common, ], 2, k2$centers[j, ])^2)))
    adjusted_rand(p1, p2) })
  cat(sprintf("stability k=%d: mean ARI between two random halves = %.3f (worst %.3f)\n", k, mean(aris), min(aris)))
}



Los índices concuerdan aquí, lo que no es el caso habitual pero resulta instructivo. La silueta y el índice de Davies–Bouldin prefieren ambos \(k = 2\); los índices externos (ARI, NMI) también alcanzan su máximo en \(k = 2\), donde los clusters se alinean mejor con la biopsia, y disminuyen a medida que \(k\) crece porque los clusters adicionales dividen los grupos diagnósticos. El análisis de estabilidad muestra que la solución de dos clusters es altamente reproducible entre mitades aleatorias (ARI medio de alrededor de 0.98) y que la estabilidad disminuye con \(k\). En este conjunto de datos, los tres tipos de evidencia concuerdan en dos clusters, pero esa concordancia es un hecho sobre estos 683 casos y estas nueve características, no un teorema.

### 7.1 La trampa: validar los clusters con las características que los construyeron

Un error común en la literatura aplicada merece su propia demostración. Tras agrupar a los pacientes según un conjunto de variables, el analista prueba si los clusters difieren en esas variables, encuentra \(p < 0.001\) en todas partes e informa que los clusters son "estadísticamente distintos". Lo son, por construcción. k-means <em>eligió</em> la partición que maximiza la separación de los grupos precisamente en esas variables, de modo que una prueba de esa separación es circular, y será significativa incluso cuando los datos no contengan ninguna estructura. El bloque de código aplica k-means a ruido puro y ejecuta las pruebas.



In [ ]:
set.seed(0)
X_noise <- matrix(rnorm(200 * 10), 200, 10)                 # 200 'pacientes', 10 'biomarcadores', sin estructura
lab_noise <- kmeans(X_noise, 2, nstart = 10)$cluster
pvals <- sapply(1:10, function(j) t.test(X_noise[lab_noise == 1, j], X_noise[lab_noise == 2, j])$p.value)
cat("cluster sizes:", as.vector(table(lab_noise)), "\n")
cat("p-values of the 10 between-cluster t-tests on the clustering features:\n", round(pvals, 4), "\n")
cat(sprintf("features 'significant' at p < 0.05: %d of 10 -- in data with no groups\n", sum(pvals < 0.05)))
X_held <- matrix(rnorm(200 * 10), 200, 10)                  # la prueba honesta: variables NO usadas para agrupar
p_held <- sapply(1:10, function(j) t.test(X_held[lab_noise == 1, j], X_held[lab_noise == 2, j])$p.value)
cat(sprintf("the same tests on 10 held-out variables: %d of 10 significant (expected about 0.5 by chance)\n", sum(p_held < 0.05)))



La mitad de los diez "biomarcadores" difieren significativamente entre dos clusters de ruido puro, al menos uno con un valor \(p\) ínfimo. El remedio está en la última línea: validar los clusters con información que no intervino en su construcción —variables reservadas, un desenlace clínico posterior, una cohorte independiente, una respuesta al tratamiento—. Los fenotipos de sepsis de la Sección 8 son un modelo de esto: derivados de variables clínicas al ingreso, se validaron por su asociación con la mortalidad y con la respuesta al tratamiento, ninguna de las cuales se usó para construirlos.

<div class="box bridge"><h4>Puente con el Suplemento S1</h4><p>La circularidad aquí es la gemela no supervisada de la fuga de datos del Suplemento S1. Allí, el conjunto de prueba estaba contaminado por información del entrenamiento; aquí, la "prueba" de los clusters está contaminada por el agrupamiento. La cura es la misma: mantener la evidencia usada para <em>construir</em> un resultado separada de la evidencia usada para <em>juzgarlo</em>.</p></div>

## 8. Aplicaciones médicas

**Subtipificación de pacientes: fenotipos de sepsis.** La sepsis es un síndrome, no una enfermedad, y los ensayos de terapias dirigidas han fracasado repetidamente en poblaciones heterogéneas. Seymour y colaboradores aplicaron k-means de consenso a 29 variables clínicas y de laboratorio disponibles de rutina, medidas dentro de las seis horas posteriores a la llegada al hospital en 20,189 pacientes, e identificaron cuatro fenotipos —α, β, γ y δ— con distintos patrones de disfunción orgánica, distintos perfiles de biomarcadores y distinta mortalidad a 28 días (desde alrededor del 2 % en α hasta alrededor del 32 % en δ); los fenotipos se reprodujeron en una segunda cohorte de 43,086 pacientes y, en simulaciones de tres ensayos ya completados, el efecto aparente del tratamiento cambiaba según la composición de fenotipos.<sup class="c"><a href="#r33">[33]</a></sup> Este es el caso de uso de la Sección 1.2 en su forma más pura: no existía ninguna etiqueta, el agrupamiento propuso una y la evidencia externa —desenlaces y respuesta al tratamiento, no las variables de agrupamiento— la validó.

**Subtipos moleculares de tumores.** El mapa de calor agrupado de la Sección 3.6 definió un campo. Perou y colaboradores aplicaron agrupamiento jerárquico a la expresión de unos 8,000 genes en 65 muestras de tumores de mama y encontraron que los tumores se repartían en grupos —más tarde denominados luminal A, luminal B, HER2-enriquecido y de tipo basal— que diferían en sus patrones de expresión y, como mostraron trabajos posteriores, en pronóstico y respuesta a la terapia;<sup class="c"><a href="#r34">[34]</a></sup> estos "subtipos intrínsecos" forman hoy parte de la práctica clínica. Un año antes, Golub y colaboradores habían mostrado en leucemias agudas que el agrupamiento no supervisado de perfiles de expresión redescubría la distinción entre leucemia mieloide aguda y leucemia linfoblástica aguda sin que se le indicara (<em>descubrimiento de clases</em>), y que un clasificador supervisado podía luego asignar nuevos pacientes a esas clases (<em>predicción de clases</em>): las dos mitades del pipeline que describen esta unidad y las anteriores.<sup class="c"><a href="#r35">[35]</a></sup>

**Estructura poblacional con PCA.** Los datos de genotipo —cientos de miles de polimorfismos de un solo nucleótido por persona— son la mayor aplicación rutinaria de PCA en biomedicina. Novembre y colaboradores calcularon las dos primeras componentes principales de los genotipos de 3,000 europeos y encontraron que el diagrama de dispersión resultante reproducía el mapa de Europa: las dos componentes eran, con una buena aproximación, la latitud y la longitud de la ascendencia de los individuos.<sup class="c"><a href="#r36">[36]</a></sup> La consecuencia práctica es que las primeras componentes principales se incluyen como covariables en todo estudio de asociación del genoma completo para evitar que la ascendencia confunda la asociación entre genotipo y enfermedad: PCA como herramienta para eliminar estructura en lugar de encontrarla.

<div class="box key"><h4>Cómo leer un artículo de aprendizaje no supervisado</h4><p>El Taller 3 te pide responder un cuestionario sobre métodos no supervisados; las preguntas que hay que hacerle a cualquier artículo que los use son las que esta unidad ha ensayado. ¿Qué características se agruparon y se estandarizaron? ¿Qué método y qué distancia, y qué forma de cluster supone ese método? ¿Cómo se eligió \(k\) y cuán sensible es el resultado a esa elección? ¿Cuán estable es la partición bajo remuestreo? Y, sobre todo, ¿qué evidencia <em>externa a las variables de agrupamiento</em> muestra que los clusters significan algo? Un artículo que valida sus clusters solo con las variables que los construyeron ha demostrado la Sección 7.1, no un descubrimiento.</p></div>

## 9. Resumen

Ahora deberías ser capaz de explicar, sin apuntes:

- Qué es el aprendizaje no supervisado, las tres tareas que comprende (agrupamiento, reducción de dimensionalidad, estimación de densidad) y por qué sus resultados no pueden validarse como los de un clasificador.
- La función objetivo de k-means —la suma de cuadrados intra-cluster— y por qué minimizarla favorece clusters redondos de tamaño y dispersión similares.
- El algoritmo de Lloyd (inicializar, asignar, actualizar, repetir), por qué cada paso disminuye el objetivo, por qué alcanza un mínimo local y qué hacen k-means++ y los reinicios al respecto.
- Las variantes: fuzzy c-means (grados de pertenencia), k-medoids (un caso real como representante; cualquier disimilitud; robusto), mini-batch (escala), k-means++ (siembra), esférico (distancia coseno), y cuándo se requiere cada una.
- Cómo elegir \(k\) con el codo, la silueta y el estadístico gap, y por qué solo el estadístico gap puede responder "un cluster".
- Los tres modos de falla de k-means (tamaños desiguales, formas no convexas, características sin escalar) y el remedio para cada uno, y por qué, en los datos de cáncer de mama de nueve características conmensurables, el remedio del escalado apenas cambia el resultado.
- Por qué la concordancia del 96 % entre los clusters de k-means y la biopsia no es una exactitud, y para qué sirve el agrupamiento cuando las etiquetas no existen.
- El algoritmo aglomerativo, las cuatro reglas de enlace y sus fórmulas, el efecto de encadenamiento del enlace simple, cómo cortar un dendrograma y qué mide la correlación cofenética.
- El mapa de calor agrupado de Eisen et al. y cómo se construye (enlace promedio, distancia de correlación, ambos ejes).
- PCA como maximización de la varianza y como mínimo error de reconstrucción; cómo calcular a mano las componentes a partir de la matriz de covarianza; el gráfico de sedimentación, las cargas y el biplot; por qué la estandarización va primero; PCA como SVD; el blanqueo.
- Por qué la varianza no es información, y para qué sirven (y para qué no) t-SNE y UMAP.
- ICA como separación ciega de fuentes: independencia frente a no correlación, la no gaussianidad como palanca, las ambigüedades de permutación y de escala, y la eliminación de artefactos anulando una componente.
- Las mezclas gaussianas como k-means blando ajustado mediante EM; DBSCAN y el agrupamiento espectral como métodos basados en conectividad.
- Los índices internos y externos, la estabilidad y la circularidad de validar clusters con las variables que los construyeron.

## 10. Autoevaluación

<details><summary>1. Un colega informa que k-means sobre 25 variables de laboratorio "alcanzó un 88 % de exactitud" al separar pacientes diabéticos de no diabéticos. Señala dos cosas que están mal en la frase.</summary>k-means no usa etiquetas, de modo que no tiene exactitud: el 88 % es la concordancia con una etiqueta tras reetiquetar los clusters, y refleja que la variación relacionada con la diabetes resulta dominar esas 25 variables. Además, el algoritmo llama a los clusters 1 y 2; la correspondencia con "diabético" se eligió para maximizar la concordancia, lo que la infla (con dos clusters, la concordancia no puede ser inferior al 50 %).</details>

<details><summary>2. Ejecuta a mano una iteración del algoritmo de Lloyd sobre los puntos \((0, 0), (0, 2), (5, 0), (5, 2)\) con centroides iniciales \((0, 0)\) y \((0, 2)\). ¿Qué ocurre y qué ilustra?</summary>Asignación: \((0,0) \to \mu_1\); \((0,2) \to \mu_2\); \((5,0)\): \(d^2 = 25\) vs \(29 \to \mu_1\); \((5,2)\): \(29\) vs \(25 \to \mu_2\). Actualización: \(\mu_1 = (2.5, 0)\), \(\mu_2 = (2.5, 2)\). La asignación siguiente no cambia nada, de modo que el algoritmo converge con \(J = 4 \times 6.25 = 25\); pero la partición \(\{(0,0),(0,2)\}, \{(5,0),(5,2)\}\) tiene \(J = 4\). Un mal inicio condujo a un mínimo local; k-means++ casi con seguridad habría elegido dos puntos iniciales muy alejados.</details>

<details><summary>3. ¿Por qué el método del codo siempre produce una curva decreciente, y por qué el estadístico gap puede devolver \(k = 1\) mientras que la silueta no?</summary>Añadir un centroide nunca puede aumentar la suma de cuadrados intra-cluster mínima (la solución anterior sigue disponible), de modo que \(J^\ast(k)\) es no creciente y llega a 0 en \(k = n\). La silueta requiere al menos dos clusters para definir \(b(i)\), así que no puede evaluar \(k = 1\); el estadístico gap compara \(\log J^\ast(k)\) con su valor en datos de referencia sin estructura y puede concluir que ningún \(k > 1\) mejora a \(k = 1\) más allá de lo que lo haría el ruido uniforme.</details>

<details><summary>4. En los datos de 30 características de la edición Python, k-means sobre las características en bruto dio un ARI mucho peor que con las estandarizadas; en los datos de 9 características de esta edición apenas cambió. Explica ambos resultados.</summary>La distancia euclidiana queda dominada por las características con el mayor rango numérico. En los datos de 30 características, una característica (<em>worst area</em>, en µm²) tenía valores en los miles y anulaba a las demás, de modo que la partición en bruto era en la práctica un umbral sobre el área. Las nueve características de mlbench son todas puntuaciones citológicas en la misma escala 1–10, así que ninguna puede dominar, y estandarizar no cambia casi nada. Estandarizar sigue siendo una buena práctica, porque no cuesta nada y protege frente a la primera situación.</details>

<details><summary>5. Clusters \(A = \{(0,0), (0,1)\}\) y \(B = \{(3,0), (4,1)\}\). Calcula \(D(A, B)\) con enlace simple, completo y promedio.</summary>Las cuatro distancias entre puntos son \(d((0,0),(3,0)) = 3\), \(d((0,0),(4,1)) = \sqrt{17} \approx 4.12\), \(d((0,1),(3,0)) = \sqrt{10} \approx 3.16\), \(d((0,1),(4,1)) = 4\). Simple: \(3\). Completo: \(4.12\). Promedio: \((3 + 4.12 + 3.16 + 4)/4 \approx 3.57\).</details>

<details><summary>6. Un dendrograma construido con enlace simple sobre datos de pacientes muestra un cluster que absorbe a casi todos y unos pocos casos aislados (singletons), se corte donde se corte. ¿Qué está ocurriendo y qué intentarías?</summary>Encadenamiento: unos pocos pacientes intermedios tienden un puente entre los grupos, y el enlace simple sigue el puente, fusionando los grupos antes de que estén completos, mientras los valores atípicos aislados quedan como casos aislados. Prueba con enlace completo, promedio o de Ward, que exigen que los clusters completos estén cerca; o elimina primero los valores atípicos; o usa DBSCAN, que etiqueta los puntos dispersos como ruido en lugar de forzarlos a entrar en clusters.</details>

<details><summary>7. Calcula a mano las componentes principales de la matriz de covarianza \(\mathbf{S} = \begin{pmatrix} 3 & 1 \\ 1 & 3 \end{pmatrix}\) y la proporción de varianza explicada por la primera.</summary>\(\det(\mathbf{S} - \lambda \mathbf{I}) = (3 - \lambda)^2 - 1 = 0\) da \(\lambda_1 = 4\), \(\lambda_2 = 2\). Para \(\lambda_1 = 4\): \(-w_1 + w_2 = 0\), de modo que \(\mathbf{w}_1 = (1, 1)/\sqrt{2}\); para \(\lambda_2 = 2\), \(\mathbf{w}_2 = (1, -1)/\sqrt{2}\). Proporción explicada por PC1: \(4 / 6 \approx 0.667\).</details>

<details><summary>8. En los datos de cáncer de mama de nueve características, la primera componente estandarizada explica alrededor del 66 % de la varianza y carga casi por igual en ocho características. ¿Qué te dice eso biológicamente?</summary>Las ocho puntuaciones citológicas están fuertemente correlacionadas de forma positiva: un núcleo maligno tiende a ser anormal en todas ellas a la vez, de modo que un único eje de "grado de anormalidad" da cuenta de dos tercios de la variación. PC1 es esencialmente el promedio de esas ocho puntuaciones, y es también el eje a lo largo del cual se separan los casos benignos y malignos; por eso una sola componente principal ya clasifica los casos con alrededor del 97 % (Figura 13).</details>

<details><summary>9. En la validación cruzada de k-NN de la Sección 4.6, el PCA se volvió a ajustar en cada partición aunque PCA no usa etiquetas. ¿Por qué es necesaria la precaución?</summary>Ajustar el PCA con todos los casos, incluida la partición de prueba, permite que los casos de prueba influyan en las componentes (sus medias y direcciones), de modo que la partición de prueba ya no es independiente del preprocesamiento: fuga de datos sin etiquetas. El efecto suele ser pequeño para PCA, pero puede ser grande para otros preprocesamientos no supervisados (selección de características por varianza, imputación), y el principio es el mismo que en el Suplemento S1: todo paso que aprende de los datos debe ver solo la partición de entrenamiento.</details>

<details><summary>10. La primera componente de ICA de un registro de EEG a veces es el artefacto de parpadeo y a veces no. ¿Por qué, y cómo se identifica en la práctica la componente del artefacto?</summary>ICA tiene una ambigüedad de permutación: el orden de las componentes es arbitrario y cambia con la inicialización. La componente del artefacto se identifica por su contenido —una forma de onda con deflexiones grandes y lentas sincronizadas con los parpadeos, un patrón espacial (la fila de la matriz de separación) concentrado en los electrodos frontales y, si está disponible, una alta correlación con el canal de electrooculograma—, no por su posición.</details>

<details><summary>11. Dos señales registradas son mezclas de dos fuentes gaussianas. ¿Puede ICA separarlas? ¿Y PCA? Explica.</summary>Ninguno de los dos. Para variables gaussianas, la no correlación es independencia, y cualquier rotación de dos gaussianas independientes da otras dos gaussianas independientes, de modo que las fuentes no son identificables: no hay no gaussianidad que ICA pueda maximizar. PCA devuelve componentes no correlacionadas, pero estas son una rotación arbitraria entre infinitas que son igualmente "independientes". ICA requiere que a lo sumo una fuente sea gaussiana.</details>

<details><summary>12. Un autor agrupa pacientes con k-means sobre una matriz \(X\), luego informa que todas las variables de \(X\) difieren significativamente entre los clusters 1 y 2 \((p < 0.001)\) y concluye que los clusters son reales. ¿Qué está mal y qué sería evidencia aceptable?</summary>Las pruebas son circulares: k-means eligió la partición que maximiza la separación de los clusters precisamente en esas columnas de \(X\), de modo que las diferencias significativas están garantizadas incluso con ruido puro (Sección 7.1). La evidencia aceptable es externa a las variables de agrupamiento: diferencias en una variable reservada o en un desenlace posterior, la reproducción de los clusters en una cohorte independiente, la estabilidad bajo remuestreo o una asociación con la respuesta al tratamiento, como en el estudio de los fenotipos de sepsis.</details>

## Glosario

<dl class="gloss">
<dt>Índice de Rand ajustado (adjusted Rand index, ARI)</dt><dd>Índice externo que compara dos particiones según los pares de casos que ambas concuerdan en agrupar o separar, corregido por azar: 0 para una partición aleatoria, 1 para particiones idénticas; invariante al reetiquetado.</dd>
<dt>Agrupamiento aglomerativo (agglomerative clustering)</dt><dd>Agrupamiento jerárquico de abajo hacia arriba: se comienza con un cluster por punto y se fusionan repetidamente los dos clusters más cercanos según una regla de enlace (R: <code>hclust</code>).</dd>
<dt>Separación ciega de fuentes (blind source separation)</dt><dd>Recuperar señales fuente desconocidas a partir de mezclas observadas sin conocer el proceso de mezcla; ICA es el método principal.</dd>
<dt>Encadenamiento (chaining)</dt><dd>La tendencia del enlace simple a fusionar clusters a través de cadenas de puntos intermedios, lo que produce clusters alargados.</dd>
<dt>Distancia / correlación cofenética (cophenetic distance / correlation)</dt><dd>La altura a la que dos puntos se unen por primera vez en un dendrograma / la correlación entre las distancias cofenéticas y las originales, que mide con qué fidelidad el árbol representa los datos (R: <code>cophenetic</code>).</dd>
<dt>Matriz de covarianza (covariance matrix)</dt><dd>La matriz simétrica \(p \times p\) de las varianzas (diagonal) y covarianzas (fuera de la diagonal) de las características; sus vectores propios son las componentes principales.</dd>
<dt>Dendrograma (dendrogram)</dt><dd>El árbol que produce el agrupamiento jerárquico; las hojas son casos, los nodos internos son fusiones y la altura de un nodo es la distancia a la que ocurrió la fusión.</dd>
<dt>Vector propio / valor propio (eigenvector / eigenvalue)</dt><dd>Un vector \(\mathbf{w}\) tal que \(\mathbf{S}\mathbf{w} = \lambda\mathbf{w}\); para una matriz de covarianza, una dirección de variación y la varianza a lo largo de ella (R: <code>eigen</code>).</dd>
<dt>Varianza explicada (explained variance)</dt><dd>La fracción \(\lambda_j / \sum_l \lambda_l\) de la varianza total que contiene la componente principal \(j\).</dd>
<dt>FastICA (FastICA)</dt><dd>Algoritmo de ICA de punto fijo que blanquea los datos y luego encuentra, una a la vez, las direcciones de máxima no gaussianidad; se implementa desde cero en la Sección 5.3.</dd>
<dt>Estadístico gap (gap statistic)</dt><dd>Criterio para \(k\) que compara el \(\log J^\ast(k)\) observado con su valor esperado en datos de referencia sin estructura; puede seleccionar \(k = 1\).</dd>
<dt>Modelo de mezcla gaussiana (Gaussian mixture model)</dt><dd>Modelo de densidad como suma ponderada de gaussianas, ajustado mediante EM; sus componentes son clusters blandos con su propia covarianza (R: <code>mclust</code>).</dd>
<dt>Análisis de componentes independientes (independent component analysis, ICA)</dt><dd>Descomposición lineal en componentes estadísticamente independientes, encontradas maximizando la no gaussianidad; determinada solo salvo permutación, signo y escala.</dd>
<dt>k-means++ (k-means++)</dt><dd>Inicialización que elige los centroides sucesivos con probabilidad proporcional a su distancia al cuadrado desde los centroides ya elegidos.</dd>
<dt>Curtosis / negentropía (kurtosis / negentropy)</dt><dd>Medidas de no gaussianidad: el cuarto momento estandarizado menos 3 / el déficit de entropía respecto de una gaussiana de igual varianza.</dd>
<dt>Enlace (linkage)</dt><dd>La regla que define la distancia entre dos clusters a partir de las distancias entre sus puntos: simple (mínimo), completo (máximo), promedio (media), Ward (aumento de la suma de cuadrados intra-cluster).</dd>
<dt>Algoritmo de Lloyd (Lloyd's algorithm)</dt><dd>El procedimiento alternante de asignación/actualización que minimiza la función objetivo de k-means hasta un mínimo local.</dd>
<dt>Carga (loading)</dt><dd>El peso de una característica original en una componente principal (un elemento del vector propio).</dd>
<dt>Medoide (medoid)</dt><dd>El punto de datos de un cluster con la menor disimilitud total respecto de los demás miembros; el representante que usa k-medoids/PAM (R: <code>cluster::pam</code>).</dd>
<dt>Análisis de componentes principales (principal component analysis, PCA)</dt><dd>Transformación lineal ortogonal a componentes no correlacionadas ordenadas por varianza; de forma equivalente, el subespacio de mínimo error de reconstrucción (R: <code>prcomp</code>).</dd>
<dt>Silueta (silhouette)</dt><dd>Para un punto, \((b - a)/\max(a, b)\), donde \(a\) es su distancia media a su propio cluster y \(b\) al otro cluster más cercano; promediada sobre los puntos, un índice interno (R: <code>cluster::silhouette</code>).</dd>
<dt>Descomposición en valores singulares (singular value decomposition, SVD)</dt><dd>Factorización \(\mathbf{X} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top\); los vectores singulares derechos son las componentes principales y \(\sigma_j^2/(n-1)\) los valores propios.</dd>
<dt>Aprendizaje no supervisado (unsupervised learning)</dt><dd>Aprender a partir de una matriz de datos sin etiquetas: encontrar grupos, resúmenes de baja dimensión o una densidad.</dd>
<dt>Blanqueo (whitening)</dt><dd>Transformar los datos a covarianza identidad (componentes no correlacionadas y de varianza unitaria); el primer paso de FastICA.</dd>
<dt>Suma de cuadrados intra-cluster (within-cluster sum of squares, WCSS)</dt><dd>\(\sum_j \sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert^2\); la función objetivo de k-means (R: <code>kmeans()$tot.withinss</code>).</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r2">Jain AK. Data clustering: 50 years beyond K-means. <em>Pattern Recognit Lett</em>. 2010;31(8):651–666. <a href="https://doi.org/10.1016/j.patrec.2009.09.011">doi:10.1016/j.patrec.2009.09.011</a></li>
<li id="r3">Xu R, Wunsch D. Survey of clustering algorithms. <em>IEEE Trans Neural Netw</em>. 2005;16(3):645–678. <a href="https://doi.org/10.1109/TNN.2005.845141">doi:10.1109/TNN.2005.845141</a></li>
<li id="r4">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r5">MacQueen J. Some methods for classification and analysis of multivariate observations. In: <em>Proc Fifth Berkeley Symp Math Stat Probab</em>, Vol. 1. University of California Press; 1967:281–297. <a href="https://projecteuclid.org/euclid.bsmsp/1200512992">projecteuclid.org/euclid.bsmsp/1200512992</a> (artículo de actas sin DOI)</li>
<li id="r6">Lloyd S. Least squares quantization in PCM. <em>IEEE Trans Inf Theory</em>. 1982;28(2):129–137. <a href="https://doi.org/10.1109/TIT.1982.1056489">doi:10.1109/TIT.1982.1056489</a></li>
<li id="r7">Bezdek JC. <em>Pattern Recognition with Fuzzy Objective Function Algorithms</em>. New York: Plenum Press; 1981. <a href="https://doi.org/10.1007/978-1-4757-0450-1">doi:10.1007/978-1-4757-0450-1</a></li>
<li id="r8">Kaufman L, Rousseeuw PJ. <em>Finding Groups in Data: An Introduction to Cluster Analysis</em>. New York: Wiley; 1990. <a href="https://doi.org/10.1002/9780470316801">doi:10.1002/9780470316801</a></li>
<li id="r9">Sculley D. Web-scale k-means clustering. In: <em>Proc 19th Int Conf World Wide Web (WWW 2010)</em>. ACM; 2010:1177–1178. <a href="https://doi.org/10.1145/1772690.1772862">doi:10.1145/1772690.1772862</a></li>
<li id="r10">Arthur D, Vassilvitskii S. k-means++: the advantages of careful seeding. In: <em>Proc 18th Annual ACM-SIAM Symp Discrete Algorithms (SODA 2007)</em>. SIAM; 2007:1027–1035. <a href="https://dl.acm.org/doi/10.5555/1283383.1283494">dl.acm.org/doi/10.5555/1283383.1283494</a> (sin DOI de CrossRef)</li>
<li id="r11">Rousseeuw PJ. Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. <em>J Comput Appl Math</em>. 1987;20:53–65. <a href="https://doi.org/10.1016/0377-0427(87)90125-7">doi:10.1016/0377-0427(87)90125-7</a></li>
<li id="r12">Tibshirani R, Walther G, Hastie T. Estimating the number of clusters in a data set via the gap statistic. <em>J R Stat Soc Ser B</em>. 2001;63(2):411–423. <a href="https://doi.org/10.1111/1467-9868.00293">doi:10.1111/1467-9868.00293</a></li>
<li id="r13">Hubert L, Arabie P. Comparing partitions. <em>J Classif</em>. 1985;2(1):193–218. <a href="https://doi.org/10.1007/BF01908075">doi:10.1007/BF01908075</a></li>
<li id="r14">Lance GN, Williams WT. A general theory of classificatory sorting strategies: 1. Hierarchical systems. <em>Comput J</em>. 1967;9(4):373–380. <a href="https://doi.org/10.1093/comjnl/9.4.373">doi:10.1093/comjnl/9.4.373</a></li>
<li id="r15">Ward JH. Hierarchical grouping to optimize an objective function. <em>J Am Stat Assoc</em>. 1963;58(301):236–244. <a href="https://doi.org/10.1080/01621459.1963.10500845">doi:10.1080/01621459.1963.10500845</a></li>
<li id="r16">Sokal RR, Rohlf FJ. The comparison of dendrograms by objective methods. <em>Taxon</em>. 1962;11(2):33–40. <a href="https://doi.org/10.2307/1217208">doi:10.2307/1217208</a></li>
<li id="r17">Eisen MB, Spellman PT, Brown PO, Botstein D. Cluster analysis and display of genome-wide expression patterns. <em>Proc Natl Acad Sci USA</em>. 1998;95(25):14863–14868. <a href="https://doi.org/10.1073/pnas.95.25.14863">doi:10.1073/pnas.95.25.14863</a></li>
<li id="r18">Aeberhard S, Forina M. Wine [dataset]. UCI Machine Learning Repository; 1991. <a href="https://doi.org/10.24432/C5PC7J">doi:10.24432/C5PC7J</a> (original de la tabla muestras × características que simula el mapa de calor al estilo de Eisen de la Sección 3.6)</li>
<li id="r19">Hotelling H. Analysis of a complex of statistical variables into principal components. <em>J Educ Psychol</em>. 1933;24(6):417–441. <a href="https://doi.org/10.1037/h0071325">doi:10.1037/h0071325</a></li>
<li id="r20">Pearson K. On lines and planes of closest fit to systems of points in space. <em>Philos Mag</em>. 1901;2(11):559–572. <a href="https://doi.org/10.1080/14786440109462720">doi:10.1080/14786440109462720</a></li>
<li id="r21">Jolliffe IT, Cadima J. Principal component analysis: a review and recent developments. <em>Philos Trans R Soc A</em>. 2016;374(2065):20150202. <a href="https://doi.org/10.1098/rsta.2015.0202">doi:10.1098/rsta.2015.0202</a></li>
<li id="r22">Alpaydin E, Kaynak C. Optical Recognition of Handwritten Digits [dataset]. UCI Machine Learning Repository; 1998. <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a> (original de los datos de imágenes que simula la visualización con PCA de la Sección 4.6)</li>
<li id="r23">van der Maaten L, Hinton G. Visualizing data using t-SNE. <em>J Mach Learn Res</em>. 2008;9:2579–2605. <a href="https://www.jmlr.org/papers/v9/vandermaaten08a.html">jmlr.org/papers/v9/vandermaaten08a.html</a> (sin DOI)</li>
<li id="r24">McInnes L, Healy J, Melville J. UMAP: uniform manifold approximation and projection for dimension reduction. <em>arXiv</em> 1802.03426; 2018. <a href="https://doi.org/10.48550/arXiv.1802.03426">doi:10.48550/arXiv.1802.03426</a></li>
<li id="r25">Comon P. Independent component analysis, a new concept? <em>Signal Process</em>. 1994;36(3):287–314. <a href="https://doi.org/10.1016/0165-1684(94)90029-9">doi:10.1016/0165-1684(94)90029-9</a></li>
<li id="r26">Bell AJ, Sejnowski TJ. An information-maximization approach to blind separation and blind deconvolution. <em>Neural Comput</em>. 1995;7(6):1129–1159. <a href="https://doi.org/10.1162/neco.1995.7.6.1129">doi:10.1162/neco.1995.7.6.1129</a></li>
<li id="r27">Hyvärinen A, Oja E. Independent component analysis: algorithms and applications. <em>Neural Netw</em>. 2000;13(4–5):411–430. <a href="https://doi.org/10.1016/S0893-6080(00)00026-5">doi:10.1016/S0893-6080(00)00026-5</a></li>
<li id="r28">Makeig S, Bell AJ, Jung T-P, Sejnowski TJ. Independent component analysis of electroencephalographic data. In: <em>Advances in Neural Information Processing Systems 8 (NIPS 1995)</em>. MIT Press; 1996:145–151. <a href="https://proceedings.neurips.cc/paper/1995">proceedings.neurips.cc/paper/1995</a> (sin DOI)</li>
<li id="r29">Jung T-P, Makeig S, Humphries C, Lee T-W, McKeown MJ, Iragui V, Sejnowski TJ. Removing electroencephalographic artifacts by blind source separation. <em>Psychophysiology</em>. 2000;37(2):163–178. <a href="https://doi.org/10.1111/1469-8986.3720163">doi:10.1111/1469-8986.3720163</a></li>
<li id="r30">Dempster AP, Laird NM, Rubin DB. Maximum likelihood from incomplete data via the EM algorithm. <em>J R Stat Soc Ser B</em>. 1977;39(1):1–38. <a href="https://doi.org/10.1111/j.2517-6161.1977.tb01600.x">doi:10.1111/j.2517-6161.1977.tb01600.x</a></li>
<li id="r31">Ester M, Kriegel H-P, Sander J, Xu X. A density-based algorithm for discovering clusters in large spatial databases with noise. In: <em>Proc 2nd Int Conf Knowledge Discovery and Data Mining (KDD-96)</em>. AAAI Press; 1996:226–231. <a href="https://www.aaai.org/Papers/KDD/1996/KDD96-037.pdf">aaai.org/Papers/KDD/1996/KDD96-037.pdf</a> (sin DOI)</li>
<li id="r32">Davies DL, Bouldin DW. A cluster separation measure. <em>IEEE Trans Pattern Anal Mach Intell</em>. 1979;PAMI-1(2):224–227. <a href="https://doi.org/10.1109/TPAMI.1979.4766909">doi:10.1109/TPAMI.1979.4766909</a></li>
<li id="r33">Seymour CW, Kennedy JN, Wang S, et al. Derivation, validation, and potential treatment implications of novel clinical phenotypes for sepsis. <em>JAMA</em>. 2019;321(20):2003–2017. <a href="https://doi.org/10.1001/jama.2019.5791">doi:10.1001/jama.2019.5791</a></li>
<li id="r34">Perou CM, Sørlie T, Eisen MB, et al. Molecular portraits of human breast tumours. <em>Nature</em>. 2000;406(6797):747–752. <a href="https://doi.org/10.1038/35021093">doi:10.1038/35021093</a></li>
<li id="r35">Golub TR, Slonim DK, Tamayo P, et al. Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. <em>Science</em>. 1999;286(5439):531–537. <a href="https://doi.org/10.1126/science.286.5439.531">doi:10.1126/science.286.5439.531</a></li>
<li id="r36">Novembre J, Johnson T, Bryc K, et al. Genes mirror geography within Europe. <em>Nature</em>. 2008;456(7218):98–101. <a href="https://doi.org/10.1038/nature07331">doi:10.1038/nature07331</a></li>
<li id="r37">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (el paquete <code>MASS</code>: <code>ginv</code>, <code>mvrnorm</code>). 4th ed. New York: Springer; 2002. <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r38">Maechler M, Rousseeuw P, Struyf A, Hubert M, Hornik K. cluster: Cluster Analysis Basics and Extensions. R package. <a href="https://doi.org/10.32614/CRAN.package.cluster">doi:10.32614/CRAN.package.cluster</a></li>
</ol>

<h4>Libros de texto y recursos</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 — Capítulo 12, PCA y agrupamiento con laboratorios en R; el complemento en R de todo el curso. PDF gratuito en statlearning.com.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009 — Capítulo 14, "Unsupervised learning": análisis de clusters, PCA, ICA, el estadístico gap. PDF gratuito.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002 — la referencia para las funciones de R usadas aquí.</li>
<li>Hyvärinen A, Karhunen J, Oja E. <em>Independent Component Analysis</em>. Wiley; 2001 — la monografía de referencia sobre ICA.</li>
<li>Kassambara A. <em>Practical Guide to Cluster Analysis in R</em>. STHDA; 2017 — <code>factoextra</code>, <code>cluster</code> y <code>NbClust</code> para los métodos de las Secciones 2–3.</li>
</ul>
